In [ ]:
"""
Brain Tumor Classification Pipeline (Notebook Version)
Uses Gemini 2.5 Flash with 20 slices per patient for classification.

Handles all 4 dataset structures:
  - Glioma      : *_t1ce.nii              → use t1ce
  - Meningioma  : *-t1c.nii.gz            → use t1c
  - Pituitary   : *_acq-CE*_T1w.nii.gz    → use contrast-enhanced T1w (CECor / CESag / CE3DNav)
  - Healthy     : healthy_XXX.nii.gz      → single file, use as-is
"""

import os
import re
import base64
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from collections import Counter
import google.generativeai as genai
import csv

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

GEMINI_API_KEY = ""   # <-- replace with your key

NUM_SLICES = 20
SLICE_AXIS = 2   # axial

# ─────────────────────────────────────────────
# SMART MODALITY FILE FINDER
# Each dataset has different naming — this handles all of them
# ─────────────────────────────────────────────

# Priority-ordered patterns per dataset type
# The finder tries each pattern in order and returns the FIRST match

MODALITY_PATTERNS = {

    # ── Glioma (BraTS 2019): BraTS19_XXXX_t1ce.nii ──
    "t1ce_brats": re.compile(r".*t1ce\.nii(\.gz)?$", re.IGNORECASE),

    # ── Meningioma (BraTS 2024): BraTS-MEN-XXXXX-t1c.nii.gz ──
    "t1c_brats2024": re.compile(r".*-t1c\.nii(\.gz)?$", re.IGNORECASE),

    # ── Pituitary: contrast-enhanced T1w variants ──
    # Priority: CECor > CESag > CE3DNavigation (CECor is highest quality coronal CE)
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",          re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",          re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$", re.IGNORECASE),

    # ── Healthy: healthy_XXX.nii.gz ──
    "healthy": re.compile(r"^healthy_\d+\.nii(\.gz)?$", re.IGNORECASE),
}

# Ordered priority list — first match wins
PATTERN_PRIORITY = [
    "t1ce_brats",
    "t1c_brats2024",
    "pituitary_cecor",
    "pituitary_cesag",
    "pituitary_ce3dnav",
    "healthy",
]

def find_best_modality_file(path: str) -> tuple:
    """
    Given a patient folder (or a direct .nii/.nii.gz file path),
    find the best available modality file.

    Returns: (file_path, modality_type) or (None, None) if not found.
    """
    p = Path(path)

    # ── Direct file path (healthy dataset passes file paths directly) ──
    if p.is_file():
        fname = p.name
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(fname):
                return str(p), key
        # Accept any .nii file passed directly
        if fname.endswith(".nii") or fname.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None

    # ── Folder path → search recursively ──
    if not p.is_dir():
        return None, None

    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))

    # Filter out segmentation / mask / derivative files
    candidates = [
        f for f in candidates
        if not any(skip in f.name.lower() for skip in [
            "seg", "mask", "label", "defacemask", "seed",
            "prediction", "groundtruth", "dwi", "adcb"
        ])
    ]

    # Try patterns in priority order
    for key in PATTERN_PRIORITY:
        pattern = MODALITY_PATTERNS[key]
        matches = [f for f in candidates if pattern.match(f.name)]
        if matches:
            return str(matches[0]), key

    # Fallback: any remaining .nii file
    if candidates:
        return str(candidates[0]), "fallback"

    return None, None


# ─────────────────────────────────────────────
# SLICE EXTRACTION
# ─────────────────────────────────────────────

def extract_evenly_spaced_slices(nifti_path: str, n_slices: int = 20, axis: int = 2):
    """
    Load NIfTI, normalize, extract n evenly-spaced slices from middle 60%.

    For pituitary volumes (non-isotropic, e.g. 512x14x512):
    automatically picks the axis with the most slices.
    """
    img  = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)

    # Auto-select best axis for thin volumes (pituitary has very few slices on one axis)
    selected_axis = axis
    if data.shape[axis] < n_slices:
        selected_axis = int(np.argmax(data.shape))
        print(f"    INFO: Volume shape {data.shape} — switching to axis {selected_axis} "
              f"(has {data.shape[selected_axis]} slices)")

    # Percentile normalization — robust to outliers
    nonzero = data[data > 0]
    if len(nonzero) > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
    else:
        p1, p99 = 0.0, 1.0

    data = np.clip(data, p1, p99)
    data = (data - p1) / (p99 - p1 + 1e-8)

    vol_size = data.shape[selected_axis]
    start    = int(vol_size * 0.20)
    end      = int(vol_size * 0.80)

    # Safety clamp
    start = max(0, start)
    end   = min(vol_size, end)
    if end - start < n_slices:
        start = 0
        end   = vol_size

    indices = np.linspace(start, end - 1, n_slices, dtype=int)

    slices = []
    for idx in indices:
        if selected_axis == 0:
            sl = data[idx, :, :]
        elif selected_axis == 1:
            sl = data[:, idx, :]
        else:
            sl = data[:, :, idx]
        slices.append(sl)

    return slices


def slice_to_png_bytes(sl: np.ndarray) -> bytes:
    """Convert 2D numpy array to PNG bytes."""
    fig, ax = plt.subplots(figsize=(3, 3), dpi=100)
    ax.imshow(np.rot90(sl), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return buf.read()


# ─────────────────────────────────────────────
# TUMOR CLASS DEFINITIONS (prompt)
# ─────────────────────────────────────────────

TUMOR_DEFINITIONS = """
You are an expert neuroradiologist. You will receive 20 MRI slices from a single patient.
The images may be T1 contrast-enhanced (T1ce/T1c) for glioma/meningioma,
contrast-enhanced T1w for pituitary, or structural MRI for healthy brains.

Your task: classify into exactly ONE of these four classes.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CLASS DEFINITIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. GLIOMA
   • Location    : INTRA-AXIAL — arises WITHIN brain parenchyma (cerebral hemispheres, brainstem)
   • Borders     : ILL-DEFINED, infiltrative — blends into surrounding tissue, no clear capsule
   • Enhancement : Heterogeneous or ring-like on T1ce; necrotic core common in HGG
   • Shape       : Irregular, extends along white matter tracts
   • Key sign    : Tumor "fingers" infiltrating normal brain

2. MENINGIOMA
   • Location    : EXTRA-AXIAL — attached to dura/meninges, OUTSIDE brain, compresses inward
   • Borders     : WELL-DEFINED, sharp smooth margin with CSF cleft between tumor and brain
   • Enhancement : INTENSE and HOMOGENEOUS — uniformly bright throughout
   • Shape       : Round/lobulated with broad dural base, hemispherical
   • Key sign    : Dural tail sign (thin enhancing tail along adjacent dura)

3. PITUITARY TUMOR (Adenoma)
   • Location    : SELLAR REGION ONLY — at the base of brain, within/above pituitary gland
   • Borders     : Well-defined, may extend into cavernous sinus or upward (suprasellar)
   • Enhancement : Variable — microadenoma = less bright than normal pituitary;
                   macroadenoma = moderate heterogeneous enhancement
   • Shape       : Round/ovoid, MIDLINE, "snowman" if suprasellar extension
   • Key sign    : Expansion of sella turcica; optic chiasm compression from below

4. HEALTHY (No Tumor)
   • No focal enhancing mass anywhere
   • Symmetric cortical sulci, normal ventricles, no midline shift
   • No abnormal enhancement in parenchyma, meninges, or sellar region

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL DIFFERENTIATORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Glioma vs Meningioma  → INSIDE brain tissue vs OUTSIDE pressing inward
• Meningioma vs Pituitary → Meningioma anywhere on dura; Pituitary ONLY at the sellar base
• Glioma vs Pituitary   → Glioma in hemispheres/brainstem; Pituitary strictly at skull base
• Any tumor vs Healthy  → Is there ANY focal abnormal enhancement or mass? No → healthy

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSTRUCTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Analyze ALL 20 slices together as a 3D case.
Slices are ordered from inferior (bottom) to superior (top) of the brain.

Reason step by step:
STEP 1 — Is there an abnormal enhancing lesion or mass? (yes/no — describe what you see)
STEP 2 — Where is it located? (intra-axial / extra-axial / sellar / none)
STEP 3 — Border characteristics? (well-defined / infiltrative / N/A)
STEP 4 — Enhancement pattern? (homogeneous / ring / heterogeneous / minimal / absent)
STEP 5 — Final classification

Respond in EXACTLY this format (no extra text before or after):
STEP1: <observation>
STEP2: <observation>
STEP3: <observation>
STEP4: <observation>
CLASSIFICATION: <glioma|meningioma|pituitary|healthy>
CONFIDENCE: <high|medium|low>
REASONING: <2-3 sentences on the key features that led to this decision>
"""


# ─────────────────────────────────────────────
# GEMINI CLASSIFICATION
# ─────────────────────────────────────────────

def classify_patient(path: str) -> dict:
    """
    Classify one patient.
    path = folder (glioma/meningioma/pituitary) OR direct .nii.gz file (healthy)
    """
    # 1. Find best modality file
    nifti_path, modality_type = find_best_modality_file(path)
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    if nifti_path is None:
        print(f"  WARNING: No suitable NIfTI file found in: {path}")
        return {
            "patient":        patient_name,
            "nifti_file":     None,
            "modality_used":  None,
            "classification": "ERROR",
            "confidence":     "N/A",
            "reasoning":      "No suitable NIfTI file found.",
            "raw_response":   ""
        }

    print(f"    File     : {Path(nifti_path).name}  [{modality_type}]")

    # 2. Extract slices
    slices = extract_evenly_spaced_slices(nifti_path, n_slices=NUM_SLICES)

    # 3. Build Gemini request
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel("gemini-2.5-flash")

    parts = [TUMOR_DEFINITIONS]
    for i, sl in enumerate(slices):
        png_bytes = slice_to_png_bytes(sl)
        parts.append(f"\n[Slice {i+1} of {NUM_SLICES}]")
        parts.append({"mime_type": "image/png", "data": png_bytes})

    # 4. API call
    try:
        response = model.generate_content(parts)
        raw = response.text
    except Exception as e:
        return {
            "patient":        patient_name,
            "nifti_file":     nifti_path,
            "modality_used":  modality_type,
            "classification": "ERROR",
            "confidence":     "N/A",
            "reasoning":      f"Gemini API error: {str(e)}",
            "raw_response":   ""
        }

    # 5. Parse structured response
    classification = "UNKNOWN"
    confidence     = "UNKNOWN"
    reasoning      = ""

    for line in raw.splitlines():
        lc = line.strip()
        if lc.upper().startswith("CLASSIFICATION:"):
            val = lc.split(":", 1)[1].strip().lower()
            for cls in ["glioma", "meningioma", "pituitary", "healthy"]:
                if cls in val:
                    classification = cls
                    break
        elif lc.upper().startswith("CONFIDENCE:"):
            val = lc.split(":", 1)[1].strip().lower()
            for conf in ["high", "medium", "low"]:
                if conf in val:
                    confidence = conf
                    break
        elif lc.upper().startswith("REASONING:"):
            reasoning = lc.split(":", 1)[1].strip()

    return {
        "patient":        patient_name,
        "nifti_file":     nifti_path,
        "modality_used":  modality_type,
        "classification": classification,
        "confidence":     confidence,
        "reasoning":      reasoning,
        "raw_response":   raw
    }


# ─────────────────────────────────────────────
# DATASET-AWARE PATIENT PATH COLLECTOR
# ─────────────────────────────────────────────

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    """
    Return list of paths (folders or files) for each patient.
    dataset_type: "glioma" | "meningioma" | "pituitary" | "healthy"
    """
    base = Path(dataset_dir)

    if dataset_type == "glioma":
        # HGG/ and LGG/ subfolders, each patient is a subfolder inside
        paths = []
        for grade in ["HGG", "LGG"]:
            grade_dir = base / grade
            if grade_dir.exists():
                paths += [str(d) for d in sorted(grade_dir.iterdir()) if d.is_dir()]
        if not paths:
            # Flat structure fallback
            paths = [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
        return paths

    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]

    elif dataset_type == "pituitary":
        paths = []
        for sub_dir in sorted(base.iterdir()):
            if sub_dir.is_dir() and sub_dir.name.startswith("sub-"):
                anat = sub_dir / "anat"
                paths.append(str(anat) if anat.exists() else str(sub_dir))
        return paths

    elif dataset_type == "healthy":
        # Each file is one patient
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])

    else:
        raise ValueError(f"Unknown dataset_type '{dataset_type}'. Use: glioma / meningioma / pituitary / healthy")


# ─────────────────────────────────────────────
# BATCH RUNNER
# ─────────────────────────────────────────────

def run_batch(dataset_dir: str, dataset_type: str, true_label: str,
              output_csv: str = "results.csv", max_patients: int = None) -> list:
    """
    Classify all patients in a dataset.

    Args:
        dataset_dir   : path to dataset root
        dataset_type  : "glioma" | "meningioma" | "pituitary" | "healthy"
        true_label    : ground truth for accuracy calculation
        output_csv    : output CSV path
        max_patients  : limit patients (e.g. 5 for quick test)
    """
    patient_paths = get_patient_paths(dataset_dir, dataset_type)

    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*60}")
    print(f"  BATCH: {dataset_type.upper()}")
    print(f"{'='*60}")
    print(f"  Patients   : {len(patient_paths)}")
    print(f"  True label : {true_label}")
    print(f"  Output     : {output_csv}")
    print(f"{'='*60}\n")

    results = []
    correct = 0

    for i, path in enumerate(patient_paths):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        print(f"  [{i+1}/{len(patient_paths)}] {name}")

        result               = classify_patient(path)
        result["true_label"] = true_label
        result["correct"]    = result["classification"] == true_label

        if result["correct"]:
            correct += 1

        tick = "CORRECT" if result["correct"] else f"WRONG (predicted: {result['classification']})"
        print(f"    Result   : {result['classification'].upper()} [{result['confidence']}]  {tick}\n")

        results.append(result)

    # Save CSV
    fieldnames = ["patient", "true_label", "classification", "confidence",
                  "correct", "modality_used", "reasoning", "nifti_file"]
    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    acc = correct / len(results) * 100 if results else 0
    print(f"\n  Accuracy    : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Predictions : {dict(Counter(r['classification'] for r in results))}")
    print(f"  Saved to    : {output_csv}\n")

    return results


# ─────────────────────────────────────────────
# SINGLE PATIENT RUNNER
# ─────────────────────────────────────────────

def run_single(path: str, true_label: str = None) -> dict:
    """
    Classify a single patient.
    path = folder (glioma/meningioma/pituitary) OR .nii.gz file (healthy)
    """
    name = Path(path).stem if Path(path).is_file() else Path(path).name
    print(f"\n{'='*55}")
    print(f"  Classifying: {name}")
    print(f"{'='*55}")

    result = classify_patient(path)

    print(f"\n  Modality       : {result['modality_used']}")
    print(f"  File used      : {Path(result['nifti_file']).name if result['nifti_file'] else 'NOT FOUND'}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")

    if true_label:
        match = "CORRECT" if result["classification"] == true_label else "WRONG"
        print(f"  True label     : {true_label}  ->  {match}")

    print(f"\n  Reasoning: {result['reasoning']}")
    print(f"\n  Full model response:")
    print(f"  {'-'*50}")
    for line in result["raw_response"].splitlines():
        print(f"  {line}")
    print(f"{'='*55}\n")

    return result


# ==============================================================================
#  EDIT THESE VARIABLES AND RUN
# ==============================================================================

RUN_MODE = "batch"   # "single"  or  "batch"

# ── Single patient mode ──────────────────────────────────────────────────────
# For glioma/meningioma/pituitary: set path to patient FOLDER
# For healthy: set path to a specific .nii.gz FILE
SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/pitutary/pituitary_tumor_dataset/sub-001/anat"
SINGLE_LABEL = "pituitary"   # set to None if you don't know the true label

# ── Batch mode ───────────────────────────────────────────────────────────────
BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/pitutary/pituitary_tumor_dataset"
BATCH_TYPE   = "pituitary"      # "glioma" | "meningioma" | "pituitary" | "healthy"
BATCH_LABEL  = "pituitary"      # ground truth label for accuracy calculation
BATCH_MAX    = 10            # set to None to run ALL patients
BATCH_OUTPUT = "pitutary_results.csv"

# ── Execute ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)
    elif RUN_MODE == "batch":
        run_batch(BATCH_DIR, BATCH_TYPE, BATCH_LABEL,
                  output_csv=BATCH_OUTPUT, max_patients=BATCH_MAX)
    else:
        print("Set RUN_MODE to 'single' or 'batch'")


  BATCH: PITUITARY
  Patients   : 10
  True label : pituitary
  Output     : pitutary_results.csv

  [1/10] anat
    File     : sub-001_acq-CECor_T1w.nii.gz  [pituitary_cecor]
    Result   : HEALTHY [low]  WRONG (predicted: healthy)

  [2/10] anat
    File     : sub-002_acq-CECor_T1w.nii.gz  [pituitary_cecor]
    Result   : ERROR [N/A]  WRONG (predicted: ERROR)

  [3/10] anat
    File     : sub-003_acq-CECor_T1w.nii  [pituitary_cecor]
    INFO: Volume shape (512, 512, 13) — switching to axis 0 (has 512 slices)
    Result   : HEALTHY [low]  WRONG (predicted: healthy)

  [4/10] anat
    File     : sub-004_acq-CECor_T1w.nii.gz  [pituitary_cecor]
    Result   : HEALTHY [low]  WRONG (predicted: healthy)

  [5/10] anat
    File     : sub-005_acq-CECor_T1w.nii.gz  [pituitary_cecor]
    Result   : HEALTHY [low]  WRONG (predicted: healthy)

  [6/10] anat
    File     : sub-006_acq-CECor_T1w.nii.gz  [pituitary_cecor]
    Result   : HEALTHY [low]  WRONG (predicted: healthy)

  [7/10] anat
    Fi

In [11]:
"""
MedGemma 27B Brain Tumor Classification Pipeline
Uses Ollama + MedGemma locally for MRI tumor classification.

SUPPORTED CLASSES:
    - glioma
    - meningioma
    - pituitary
    - healthy

OUTPUT:
    ONLY final classification label.

REQUIREMENTS:
    ollama pull medgemma:27b
    pip install nibabel matplotlib pillow requests numpy

RUN:
    python medgemma_brain_classifier.py
"""

import os
import re
import json
import base64
import requests
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from io import BytesIO
from pathlib import Path
from PIL import Image

# ============================================================
# CONFIG
# ============================================================

OLLAMA_MODEL = "medgemma:27b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

NUM_SLICES = 14
SLICE_AXIS = 2

# ============================================================
# FILE FINDER
# ============================================================

MODALITY_PATTERNS = {

    # Glioma
    "t1ce_brats":
        re.compile(r".*t1ce\.nii(\.gz)?$", re.IGNORECASE),

    # Meningioma
    "t1c_brats2024":
        re.compile(r".*-t1c\.nii(\.gz)?$", re.IGNORECASE),

    # Pituitary
    "pituitary_cecor":
        re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$", re.IGNORECASE),

    "pituitary_cesag":
        re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$", re.IGNORECASE),

    "pituitary_ce3dnav":
        re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$", re.IGNORECASE),

    # Healthy
    "healthy":
        re.compile(r"^healthy_\d+\.nii(\.gz)?$", re.IGNORECASE),
}

PATTERN_PRIORITY = [
    "t1ce_brats",
    "t1c_brats2024",
    "pituitary_cecor",
    "pituitary_cesag",
    "pituitary_ce3dnav",
    "healthy",
]


def find_best_modality_file(path):

    p = Path(path)

    # direct file
    if p.is_file():

        fname = p.name

        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(fname):
                return str(p), key

        if fname.endswith(".nii") or fname.endswith(".nii.gz"):
            return str(p), "direct_file"

        return None, None

    # folder
    if not p.is_dir():
        return None, None

    candidates = sorted(
        list(p.rglob("*.nii")) +
        list(p.rglob("*.nii.gz"))
    )

    # remove masks
    candidates = [
        f for f in candidates
        if not any(skip in f.name.lower() for skip in [
            "seg", "mask", "label",
            "prediction", "groundtruth",
            "seed", "deface"
        ])
    ]

    for key in PATTERN_PRIORITY:

        pattern = MODALITY_PATTERNS[key]

        matches = [
            f for f in candidates
            if pattern.match(f.name)
        ]

        if matches:
            return str(matches[0]), key

    if candidates:
        return str(candidates[0]), "fallback"

    return None, None


# ============================================================
# MRI PREPROCESSING
# ============================================================

def normalize_volume(data):

    nonzero = data[data > 0]

    if len(nonzero) > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
    else:
        p1, p99 = 0, 1

    data = np.clip(data, p1, p99)
    data = (data - p1) / (p99 - p1 + 1e-8)

    return data


def extract_slices(nifti_path, n_slices=20, axis=2):

    img = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)

    # auto choose axis for thin volumes
    selected_axis = axis

    if data.shape[axis] < n_slices:
        selected_axis = int(np.argmax(data.shape))

    data = normalize_volume(data)

    vol_size = data.shape[selected_axis]

    start = int(vol_size * 0.20)
    end   = int(vol_size * 0.80)

    if end - start < n_slices:
        start = 0
        end   = vol_size

    indices = np.linspace(
        start,
        end - 1,
        n_slices,
        dtype=int
    )

    slices = []

    for idx in indices:

        if selected_axis == 0:
            sl = data[idx, :, :]

        elif selected_axis == 1:
            sl = data[:, idx, :]

        else:
            sl = data[:, :, idx]

        slices.append(sl)

    return slices


# ============================================================
# IMAGE CONVERSION
# ============================================================

def slice_to_base64(sl):

    sl = np.rot90(sl)

    fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

    ax.imshow(sl, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

    buf = BytesIO()

    plt.savefig(
        buf,
        format="png",
        bbox_inches="tight",
        pad_inches=0
    )

    plt.close(fig)

    buf.seek(0)

    return base64.b64encode(buf.read()).decode("utf-8")


# ============================================================
# ROBUST RADIOLOGY PROMPT
# ============================================================
PROMPT = """
### TASK
Classify the brain tumor in the attached MRI slices into one of the following categories: [Glioma, Meningioma, Pituitary Tumor, or No Tumor].

### ANALYSIS PROTOCOL
1. **Anatomical Localization:** Identify the lobe or region (e.g., intra-axial vs. extra-axial).
2. **Signal Intensity:** Describe the appearance on T1, T2, and FLAIR (if provided).
3. **Mass Effect:** Note any midline shift, ventricular compression, or peritumoral edema.
4. **Enhancement Pattern:** Describe the contrast uptake (e.g., ring-enhancing, homogenous).

### IMAGE CONTEXT
- Modality: MRI (T1-weighted Contrast Enhanced)
- View: Axial and Coronal
- Patient History: [Insert Brief History if available, e.g., 17Y/M with persistent headaches]

### OUTPUT FORMAT
- **Findings:** [Bullet points]
- **Differential Diagnosis:** [Primary and Secondary]
- **Final Classification:** [Category]
- **Confidence Score:** [0-100%]
"""
# ============================================================
# OLLAMA MEDGEMMA CLASSIFIER
# ============================================================

def classify_patient(path):

    nifti_path, modality = find_best_modality_file(path)

    if nifti_path is None:
        return "error"

    print(f"\nUsing file: {Path(nifti_path).name}")
    print(f"Modality  : {modality}")

    slices = extract_slices(
        nifti_path,
        n_slices=NUM_SLICES,
        axis=SLICE_AXIS
    )

    images_b64 = []

    for sl in slices:
        images_b64.append(slice_to_base64(sl))

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": PROMPT,
        "images": images_b64,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 0.1,
            "num_predict": 5
        }
    }

    try:

        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=600
        )

        result = response.json()["response"].strip().lower()

        # sanitize
        for cls in [
            "glioma",
            "meningioma",
            "pituitary",
            "healthy"
        ]:
            if cls in result:
                return cls

        return result

    except Exception as e:
        print("ERROR:", e)
        return "error"


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # EXAMPLE PATH
    PATIENT_PATH = "/home/23bq1a4293/Personal/project/seg_data/pitutary/pituitary_tumor_dataset/sub-001/anat"

    prediction = classify_patient(PATIENT_PATH)

    print("\n====================================")
    print("FINAL CLASSIFICATION:", prediction.upper())
    print("====================================")


Using file: sub-001_acq-CECor_T1w.nii.gz
Modality  : pituitary_cecor

FINAL CLASSIFICATION: **FINDINGS:**
*


In [38]:
import os
import re
import json
import base64
import requests
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from io import BytesIO
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
OLLAMA_MODEL = "medgemma:27b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# Higher slice count allows MedGemma to see the tumor's extent
NUM_SLICES = 50
SLICE_AXIS = 2

# ============================================================
# IMPROVED RADIOLOGY PROMPT
# ============================================================
# Using a structured prompt prevents the model from giving one-word 
# answers that lack medical context/accuracy.
PROMPT = """
### CLINICAL ROLE
You are an expert AI Neuroradiologist. Your task is to classify MRI slices by mapping visual features to specific international consortium standards: BraTS 2013 (Glioma), BraTS-MEN (Meningioma), and the Černý Multiparametric standards (Pituitary).

### STEP 1: EXCLUSION OF NORMAL ANATOMY (HEALTHY GATE)
Before classifying a mass, confirm the findings are NOT normal anatomy:
- Falx Cerebri/Tentorium? (Normal thin, midline dural reflections).
- Normal Pituitary Gland? (Pea-sized, intensely and uniformly enhancing, no expansion of the sella).
- Normal Symmetry? (Symmetrical hemispheres, no ventricular compression).
*If YES to these and NO mass effect -> HEALTHY.*

### STEP 2: VISUAL FEATURE EXTRACTION
If a lesion is present, explicitly identify which of these feature sets apply:

[Glioma Features - BraTS 2013]
- ET (Enhancing Tumor): Bright contrast uptake (often ring-like).
- NCR (Necrotic Core): Dark, non-enhancing center within the ET.
- ED (Peritumoral Edema): Swelling/hyperintensity causing mass effect.

[Meningioma Features - BraTS-MEN]
- Homogenous ET: Solid, uniform bright enhancement.
- Dural Tail: Tapering enhancement along the meninges.
- Extra-axial Base: Abutting the skull, falx, or tentorium.

[Pituitary Features - Černý]
- Sellar Expansion: Mass located within and expanding the sella turcica (often a "snowman" shape).
- Differential Enhancement: The adenoma appears hypo-enhancing (darker) compared to the intensely bright normal pituitary gland on T1c.
- Compression/Invasion: Upward compression of the Optic Chiasm or lateral invasion toward the Internal Carotid Arteries (ICAs).

### STEP 3: DIFFERENTIAL DIAGNOSIS 
Select the classification based strictly on the extracted features:
- GLIOMA: Intra-axial. Requires ET (ring), NCR, and ED.
- MENINGIOMA: Extra-axial. Requires Homogenous ET, dural connection/tail, and lacks a nested necrotic core.
- PITUITARY: Sellar/Parasellar. Requires sellar expansion and differential hypo-enhancement.
- HEALTHY: Lacks all above pathological mass features.

### OUTPUT FORMAT (STRICT)
Anatomical Location: [State explicitly: Intra-axial, Extra-axial, or Sellar]
Extracted Features: [List the specific features found from Step 2, or "None"]
Analysis: [One sentence linking the location and features to the classification]
FINAL CLASSIFICATION: [GLIOMA, MENINGIOMA, PITUITARY, or HEALTHY]
Confidence: [0-100%]
"""

# ============================================================
# MRI PREPROCESSING
# ============================================================

def normalize_volume(data):
    nonzero = data[data > 0]
    if len(nonzero) > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
    else:
        p1, p99 = 0, 1
    data = np.clip(data, p1, p99)
    data = (data - p1) / (p99 - p1 + 1e-8)
    return data

def extract_slices(nifti_path, n_slices=16, axis=1):
    img = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)
    
    # Auto-fallback for orientation
    selected_axis = axis if data.shape[axis] >= n_slices else int(np.argmax(data.shape))
    data = normalize_volume(data)
    vol_size = data.shape[selected_axis]

    # Focus on the middle 60% of the brain where tumors usually reside
    indices = np.linspace(vol_size * 0.2, vol_size * 0.8, n_slices, dtype=int)
    
    slices = []
    for idx in indices:
        if selected_axis == 0: sl = data[idx, :, :]
        elif selected_axis == 1: sl = data[:, idx, :]
        else: sl = data[:, :, idx]
        slices.append(sl)
    return slices

def slice_to_base64(sl):
    sl = np.rot90(sl)
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.imshow(sl, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    
    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

# ============================================================
# CLASSIFICATION LOGIC
# ============================================================

def classify_patient(path):
    # (Reuse your find_best_modality_file logic here)
    # For this example, assuming nifti_path is resolved:
    nifti_path = path 
    
    print(f"\nProcessing: {Path(nifti_path).name}")
    slices = extract_slices(nifti_path, n_slices=NUM_SLICES)
    images_b64 = [slice_to_base64(sl) for sl in slices]

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": PROMPT,
        "images": images_b64,
        "stream": False,
        "options": {
            "temperature": 0.0, # Deterministic for medical tasks
            "num_predict": 512, # Enough tokens for reasoning + label
            "top_k": 20,
            "top_p": 0.9
        }
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600)
        full_text = response.json().get("response", "")
        
        print("\n--- MEDGEMMA ANALYSIS ---")
        print(full_text)
        print("--------------------------")

        # Robust label extraction
        cleaned_text = full_text.upper()
        labels = ["GLIOMA", "MENINGIOMA", "PITUITARY", "HEALTHY"]
        
        # Look specifically for the "Final Classification" line
        match = re.search(r"FINAL CLASSIFICATION:\s*(\w+)", cleaned_text)
        if match:
            predicted = match.group(1)
            if predicted in labels: return predicted

        # Fallback: scan whole text if formatting was missed
        for label in labels:
            if label in cleaned_text:
                return label

        return "UNKNOWN"

    except Exception as e:
        return f"ERROR: {str(e)}"

if __name__ == "__main__":
    # Update this to your actual test file
    TEST_FILE = "/home/23bq1a4293/Personal/project/seg_data/pitutary/pituitary_tumor_dataset/sub-007/anat/sub-007_acq-CECor_T1w.nii.gz"
    
    if os.path.exists(TEST_FILE):
        prediction = classify_patient(TEST_FILE)
        print(f"\nRESULT: {prediction}")
    else:
        print("File not found.")


Processing: sub-007_acq-CECor_T1w.nii.gz

--- MEDGEMMA ANALYSIS ---
Anatomical Location: Intra-axial
Extracted Features: ET, NCR, ED
Analysis: The lesion is intra-axial, exhibiting enhancing tumor (ET), a necrotic core (NCR), and peritumoral edema (ED), consistent with a glioma.
FINAL CLASSIFICATION: GLIOMA
Confidence: 95%
--------------------------

RESULT: GLIOMA


In [44]:
"""
MedGemma 27B Multi-Oncology Brain Tumor Classification Pipeline
Integrates BraTS 2013 (Glioma), BraTS-MEN (Meningioma), and Černý 2025 (Pituitary).

SUPPORTED CLASSES:
    - GLIOMA
    - MENINGIOMA
    - PITUITARY
    - HEALTHY

REQUIREMENTS:
    ollama pull medgemma:27b
    pip install nibabel matplotlib pillow requests numpy
"""

import os
import re
import json
import base64
import requests
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from io import BytesIO
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

OLLAMA_MODEL = "medgemma:27b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

NUM_SLICES = 16
DEFAULT_AXIS = 2

# ============================================================
# FILE FINDER
# ============================================================

MODALITY_PATTERNS = {
    # Glioma
    "t1ce_brats": re.compile(r".*t1ce\.nii(\.gz)?$", re.IGNORECASE),
    # Meningioma
    "t1c_brats2024": re.compile(r".*-t1c\.nii(\.gz)?$", re.IGNORECASE),
    # Pituitary
    "pituitary_cecor": re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$", re.IGNORECASE),
    "pituitary_cesag": re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$", re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$", re.IGNORECASE),
    # Healthy
    "healthy": re.compile(r"^healthy_\d+\.nii(\.gz)?$", re.IGNORECASE),
}

PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024", "pituitary_cecor", 
    "pituitary_cesag", "pituitary_ce3dnav", "healthy"
]

def find_best_modality_file(path):
    p = Path(path)
    if p.is_file():
        fname = p.name
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(fname): return str(p), key
        if fname.endswith(".nii") or fname.endswith(".nii.gz"): return str(p), "direct_file"
        return None, None

    if not p.is_dir(): return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates if not any(skip in f.name.lower() for skip in ["seg", "mask", "label", "prediction", "groundtruth", "seed", "deface"])]
    
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches: return str(matches[0]), key

    if candidates: return str(candidates[0]), "fallback"
    return None, None

# ============================================================
# MRI PREPROCESSING (Intelligent Axis & Sellar Crop)
# ============================================================

def normalize_volume(data):
    nonzero = data[data > 0]
    if len(nonzero) > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
    else:
        p1, p99 = 0, 1
    data = np.clip(data, p1, p99)
    data = (data - p1) / (p99 - p1 + 1e-8)
    return data

def extract_slices(nifti_path, modality_str, n_slices=16, default_axis=2):
    img = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)

    # INTELLIGENT AXIS SELECTION: Fallback to largest dimension if default axis is too thin
    selected_axis = default_axis
    if data.shape[default_axis] < n_slices:
        selected_axis = int(np.argmax(data.shape))

    data = normalize_volume(data)
    vol_size = data.shape[selected_axis]

    # DYNAMIC SLICING TARGET based on modality
    if modality_str and "pituitary" in modality_str.lower():
        # Target the Sella Turcica (middle 40-60%)
        start, end = int(vol_size * 0.40), int(vol_size * 0.60)
        do_crop = True 
    else:
        # Standard brain scan (middle 20-80%)
        start, end = int(vol_size * 0.20), int(vol_size * 0.80)
        do_crop = False

    if end - start < n_slices:
        start, end = 0, vol_size

    indices = np.linspace(start, end - 1, n_slices, dtype=int)
    slices = []

    for idx in indices:
        if selected_axis == 0: sl = data[idx, :, :]
        elif selected_axis == 1: sl = data[:, idx, :]
        else: sl = data[:, :, idx]

        # Sellar Crop: Zoom in heavily on the skull base for Pituitary scans
        if do_crop:
            h, w = sl.shape
            crop_h, crop_w = int(h * 0.25), int(w * 0.25)
            # Failsafe to prevent cropping tiny images to 0
            if h > 64 and w > 64: 
                sl = sl[crop_h:-crop_h, crop_w:-crop_w]

        slices.append(sl)

    return slices

def slice_to_base64(sl):
    sl = np.rot90(sl)
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.imshow(sl, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
    
    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

# ============================================================
# MASTER PROMPT 
# ============================================================

BASE_PROMPT = """
### CLINICAL ROLE
You are an expert AI Neuroradiologist. You must classify MRI slices using strict Anatomical Location tracking and international feature standards (BraTS 2013, BraTS-MEN, Černý).

### STEP 1: ANATOMICAL LOCATION LOCK (MANDATORY)
Identify the primary location of the suspected region:
- INTRA-AXIAL: Inside the brain parenchyma (cerebrum, cerebellum, brainstem).
- EXTRA-AXIAL: Outside the brain, abutting the skull or dural reflections.
- SELLAR: Strictly located in the sella turcica / pituitary fossa at the skull base.
- NORMAL: No distinct mass, normal symmetry.

### STEP 2: THE HEALTHY VETO GATES (CRITICAL)
Before diagnosing a tumor, you MUST pass these vetoes:
- MENINGIOMA VETO: Is the enhancement purely linear along the midline? If yes, that is the normal Falx Cerebri. A Meningioma requires a distinct nodular/globular mass.
- PITUITARY VETO: Is the gland pea-sized and uniformly bright? If yes, it is a normal gland. A Pituitary tumor requires sellar expansion and differential (darker) enhancement.

### STEP 3: FEATURE EXTRACTION & CLASSIFICATION
Map the features based ONLY on the Location Lock:
- If INTRA-AXIAL -> Evaluate for GLIOMA. (STRICTLY Intra-axial. Requires Enhancing ring, Necrotic core, and Peritumoral edema. Do NOT classify sellar masses as Gliomas).
- If EXTRA-AXIAL -> Evaluate for MENINGIOMA. (Requires solid nodular mass, dural tail, and homogenous enhancement. Do NOT classify the normal falx cerebri as Meningioma).
- If SELLAR -> Evaluate for PITUITARY. (STRICTLY Sellar. Mass expands the sella turcica. It may visually mimic a glioma, but its location at the skull base overrides other features).
- If NORMAL or VETOED -> HEALTHY.

### OUTPUT FORMAT (STRICT)
Anatomical Location: [Intra-axial, Extra-axial, Sellar, or Normal]
Extracted Features: [List the specific features found, or "None"]
Analysis: [One sentence linking the location and features to the classification]
FINAL CLASSIFICATION: [GLIOMA, MENINGIOMA, PITUITARY, or HEALTHY]
Confidence: [0-100%]
"""

# ============================================================
# OLLAMA MEDGEMMA CLASSIFIER
# ============================================================

def classify_patient(path):
    nifti_path, modality = find_best_modality_file(path)
    if nifti_path is None: return "error"

    print(f"\nProcessing: {Path(nifti_path).name}")
    print(f"Detected Modality: {modality}")

    slices = extract_slices(nifti_path, modality_str=modality, n_slices=NUM_SLICES, default_axis=DEFAULT_AXIS)
    images_b64 = [slice_to_base64(sl) for sl in slices]

    # Inject metadata directly into the prompt to anchor the LLM
    dynamic_prompt = f"### SCAN METADATA (CRITICAL CONTEXT)\n- Pipeline Modality Detected: {modality.upper()}\n- Note: If modality indicates 'PITUITARY', the lesion originates in the sella turcica. Do NOT classify suprasellar extensions as Intra-axial Gliomas.\n\n" + BASE_PROMPT

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": dynamic_prompt,
        "images": images_b64,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 0.1,
            "num_predict": 512
        }
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600)
        full_text = response.json().get("response", "")
        
        print("\n--- MEDGEMMA ANALYSIS ---")
        print(full_text.strip())
        print("--------------------------")

        # Robust label extraction
        cleaned_text = full_text.upper()
        labels = ["GLIOMA", "MENINGIOMA", "PITUITARY", "HEALTHY"]
        
        match = re.search(r"FINAL CLASSIFICATION:\s*(\w+)", cleaned_text)
        if match:
            predicted = match.group(1)
            if predicted in labels: return predicted

        for label in labels:
            if label in cleaned_text: return label

        return "UNKNOWN"

    except Exception as e:
        return f"ERROR: {str(e)}"

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    # Point this to your test directory or file
    PATIENT_PATH = "/home/23bq1a4293/Personal/project/seg_data/healthy/healthy_003.nii.gz"
    
    if os.path.exists(PATIENT_PATH):
        prediction = classify_patient(PATIENT_PATH)
        print(f"\nRESULT: {prediction}")
    else:
        print("File not found.")


Processing: healthy_003.nii.gz
Detected Modality: healthy

--- MEDGEMMA ANALYSIS ---
### STEP 1: ANATOMICAL LOCATION LOCK
The images show axial views of the brain. The region of interest appears to be located within the sella turcica, a bony depression at the base of the skull that houses the pituitary gland.

### STEP 2: THE HEALTHY VETO GATES
- MENINGIOMA VETO: The enhancement pattern is not linear along the midline falx cerebri.
- PITUITARY VETO: The structure within the sella turcica appears to be a mass expanding the sella, not a normal pea-sized gland.

### STEP 3: FEATURE EXTRACTION & CLASSIFICATION
Based on the location within the sella turcica, the primary classification to consider is PITUITARY. The images show a mass expanding the sella turcica, which is characteristic of a pituitary adenoma.

Anatomical Location: Sellar
Extracted Features: Mass expanding the sella turcica.
Analysis: The lesion is located within the sella turcica and expands it, consistent with a pituitary 

In [1]:
"""
Med3DVLM Brain Tumor Classification
=====================================
STEP 1 — Get weights (run this ONCE in terminal from inside Med3DVLM folder):

    cd /path/to/Med3DVLM
    python download_med3dvlm.py

  If that script fails, use this manual download instead:

    python -c "
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='MagicXin/Med3DVLM-Qwen-2.5-7B',
    local_dir='./downloaded_models/Med3DVLM'
)
print('Done')
"

STEP 2 — Set PYTHONPATH (run in terminal before running this script):

    export PYTHONPATH=/path/to/Med3DVLM:$PYTHONPATH

STEP 3 — Install requirements:

    cd /path/to/Med3DVLM
    pip install -r requirements.txt

STEP 4 — Edit paths below and run this script.

Datasets handled:
  Glioma      : *_t1ce.nii          (BraTS 2019)
  Meningioma  : *-t1c.nii.gz        (BraTS 2024)
  Pituitary   : *acq-CECor_T1w*     (BIDS)
  Healthy     : healthy_XXX.nii.gz  (flat)
"""

import os
import re
import sys
import csv
import warnings
import numpy as np
import nibabel as nib

from pathlib import Path
from collections import Counter
from datetime import datetime
from scipy.ndimage import zoom
from tqdm import tqdm

import torch

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE PATHS
# ══════════════════════════════════════════════════════════════════

MED3DVLM_REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"  # update if different
MED3DVLM_WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"
DEVICE           = "cuda:0" if torch.cuda.is_available() else "cpu"

# ── Input shape expected by Med3DVLM (from M3D dataset standard) ──
TARGET_D, TARGET_H, TARGET_W = 32, 256, 256

# ══════════════════════════════════════════════════════════════════
#  PYTHONPATH SETUP  (done automatically — no need to set manually)
# ══════════════════════════════════════════════════════════════════

def setup_paths():
    """Add repo src to path if it exists — not required for MagicXin weights."""
    repo = Path(MED3DVLM_REPO).resolve()
    if repo.exists():
        for p in [str(repo), str(repo / "src")]:
            if p not in sys.path:
                sys.path.insert(0, p)
        print(f"  Repo path : {repo} added to sys.path")
    else:
        print(f"  Note: Repo not found at {repo} — using trust_remote_code from weights")

setup_paths()

# ══════════════════════════════════════════════════════════════════
#  MODEL LOADER
# ══════════════════════════════════════════════════════════════════

_model     = None
_tokenizer = None

def load_model():
    global _model, _tokenizer

    if _model is not None:
        print("  Model already loaded — reusing")
        return _model, _tokenizer

    weights = Path(MED3DVLM_WEIGHTS).resolve()
    if not weights.exists():
        raise FileNotFoundError(
            f"\nWeights not found at: {weights}"
            f"\nDownload with:"
            f"\n  python -c \"from huggingface_hub import snapshot_download; "
            f"snapshot_download(repo_id='MagicXin/Med3DVLM-Qwen-2.5-7B', "
            f"local_dir='{weights}')\"  "
        )

    modeling_py = weights / "modeling.py"
    if not modeling_py.exists():
        raise FileNotFoundError(
            f"modeling.py not found inside weights folder: {weights}\n"
            f"Expected: {modeling_py}"
        )

    print(f"\n{'='*60}")
    print(f"  Loading Med3DVLM")
    print(f"  Weights    : {weights}")
    print(f"  modeling.py: FOUND ✓")
    print(f"  Device     : {DEVICE}")
    print(f"{'='*60}")

    # The weights folder contains modeling.py with trust_remote_code
    # This is the correct loading pattern for MagicXin/Med3DVLM-Qwen-2.5-7B
    from transformers import AutoTokenizer, AutoModelForCausalLM

    _tokenizer = AutoTokenizer.from_pretrained(
        str(weights),
        trust_remote_code=True,
        use_fast=False,
        padding_side="right",
    )

    # Add special image token if not present
    if "<image>" not in _tokenizer.get_vocab():
        _tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
        print("  Added <image> token to tokenizer")

    _model = AutoModelForCausalLM.from_pretrained(
        str(weights),
        torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        trust_remote_code=True,   # loads modeling.py from weights folder
    ).eval()

    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  VRAM     : {used:.1f}GB used | {total - used:.1f}GB free")

    print(f"  Model ready ✓")
    return _model, _tokenizer


# ══════════════════════════════════════════════════════════════════
#  MODALITY FILE FINDER
# ══════════════════════════════════════════════════════════════════

MODALITY_PATTERNS = {
    "t1ce_brats":        re.compile(r".*t1ce\.nii(\.gz)?$",                    re.IGNORECASE),
    "t1c_brats2024":     re.compile(r".*-t1c\.nii(\.gz)?$",                    re.IGNORECASE),
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$",  re.IGNORECASE),
    "healthy":           re.compile(r"^healthy_\d+\.nii(\.gz)?$",              re.IGNORECASE),
}
PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024",
    "pituitary_cecor", "pituitary_cesag", "pituitary_ce3dnav", "healthy",
]
SKIP_KEYWORDS = [
    "seg", "mask", "label", "defacemask", "seed",
    "prediction", "groundtruth", "dwi", "adcb"
]

def find_best_modality_file(path: str) -> tuple:
    p = Path(path)
    if p.is_file():
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(p.name):
                return str(p), key
        if p.name.endswith(".nii") or p.name.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None
    if not p.is_dir():
        return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates
                  if not any(kw in f.name.lower() for kw in SKIP_KEYWORDS)]
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches:
            return str(matches[0]), key
    return (str(candidates[0]), "fallback") if candidates else (None, None)


# ══════════════════════════════════════════════════════════════════
#  PREPROCESSING  →  tensor shape [1, 1, 32, 256, 256]
# ══════════════════════════════════════════════════════════════════

def preprocess_volume(nifti_path: str) -> torch.Tensor:
    """
    Load NIfTI → detect axial axis from affine → normalize [0,1]
    → resample to 32×256×256 → return tensor [1,1,D,H,W]
    """
    img  = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)
    if data.ndim == 4:
        data = data[..., 0]

    # Detect true axial (Z) axis from NIfTI affine
    z_row      = np.abs(img.affine[2, :3])
    axial_axis = int(np.argmax(z_row))

    # Reorder axes so axial = first dim (D, H, W)
    axes = [axial_axis] + [i for i in range(3) if i != axial_axis]
    data = np.transpose(data, axes)

    # Percentile normalization — robust to outliers
    nonzero = data[data > 0]
    if len(nonzero) > 100:
        p1  = float(np.percentile(nonzero, 1))
        p99 = float(np.percentile(nonzero, 99))
        data = np.clip(data, p1, p99)
        data = (data - p1) / (p99 - p1 + 1e-8)
    else:
        mn, mx = data.min(), data.max()
        data = (data - mn) / (mx - mn + 1e-8)
    data = data.clip(0.0, 1.0)

    # Resample to (32, 256, 256)
    d, h, w = data.shape
    factors = (TARGET_D / d, TARGET_H / h, TARGET_W / w)
    data    = zoom(data, factors, order=1)

    # → tensor [1, 1, D, H, W]
    tensor = torch.from_numpy(data).float()
    tensor = tensor.unsqueeze(0).unsqueeze(0)   # batch + channel
    return tensor


# ══════════════════════════════════════════════════════════════════
#  PROMPT
# ══════════════════════════════════════════════════════════════════

CLASSIFICATION_QUESTION = (
    "You are a consultant neuroradiologist analyzing a 3D brain MRI volume "
    "(T1 contrast-enhanced). "
    "Classify this case into EXACTLY ONE of four classes: "
    "glioma | meningioma | pituitary | healthy\n\n"

    "APPLY THIS TEST FIRST:\n"
    "  Does the enhancing mass TOUCH the inner skull surface? "
    "YES → meningioma. NO → check if inside brain (glioma) or at skull base (pituitary).\n\n"

    "GLIOMA: Inside brain parenchyma. Fuzzy infiltrative borders. "
    "Ring enhancement + dark necrotic center (HGG) or absent enhancement (LGG).\n"
    "MENINGIOMA: Outside brain on dura/skull surface. Sharp clean border. "
    "Solid homogeneous bright enhancement. Dural tail sign.\n"
    "PITUITARY: Midline skull base only in sella turcica. "
    "Snowman shape (macroadenoma) or dark spot in bright gland (microadenoma).\n"
    "HEALTHY: No focal mass. Normal symmetric brain. "
    "Normal pituitary enhancement is NOT a tumor.\n\n"

    "RESPOND IN EXACTLY THIS FORMAT — NO OTHER TEXT:\n"
    "STEP1: <skull contact test result>\n"
    "STEP2: <precise location — intra-axial / extra-axial / sellar / none>\n"
    "STEP3: <border type — sharp or infiltrative>\n"
    "STEP4: <enhancement — homogeneous solid / ring+necrosis / minimal / absent>\n"
    "CLASSIFICATION: <glioma|meningioma|pituitary|healthy>\n"
    "CONFIDENCE: <high|medium|low>\n"
    "REASONING: <3 sentences citing 3D spatial features and ruling out the runner-up class>"
)


# ══════════════════════════════════════════════════════════════════
#  INFERENCE
# ══════════════════════════════════════════════════════════════════

def run_inference(volume_tensor: torch.Tensor, model, tokenizer) -> str:
    """
    Run Med3DVLM inference.
    volume_tensor shape: [1, 1, 32, 256, 256]  float32  values [0, 1]

    MagicXin/Med3DVLM-Qwen-2.5-7B has:
      - vision_encoder   : encodes 3D volume → visual features
      - mm_projector     : projects visual features → LLM token space
      - LLM backbone     : Qwen2.5-7B-Instruct
    The model.generate() call accepts images= kwarg as per modeling.py
    """
    volume_tensor = volume_tensor.to(DEVICE, dtype=torch.bfloat16)

    # Build prompt with <image> placeholder token
    image_token_id = tokenizer.convert_tokens_to_ids("<image>")
    prompt = f"<image>\n{CLASSIFICATION_QUESTION}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768,
        padding=True,
    )
    input_ids      = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    try:
        # Primary: use images= kwarg (standard Med3DVLM interface)
        with torch.inference_mode():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                images=volume_tensor,
                max_new_tokens=300,
                do_sample=False,
                use_cache=True,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id,
            )
    except TypeError:
        # Fallback: some versions use pixel_values= instead of images=
        with torch.inference_mode():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=volume_tensor,
                max_new_tokens=300,
                do_sample=False,
                use_cache=True,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id,
            )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    del volume_tensor, input_ids, attention_mask, output_ids
    torch.cuda.empty_cache()

    return raw


# ══════════════════════════════════════════════════════════════════
#  RESPONSE PARSER
# ══════════════════════════════════════════════════════════════════

def parse_response(raw: str) -> dict:
    classification = "UNKNOWN"
    confidence     = "UNKNOWN"
    reasoning      = ""
    steps          = {}

    lines = raw.splitlines()
    idx   = 0
    while idx < len(lines):
        lc = lines[idx].strip()
        if not lc or ":" not in lc:
            idx += 1
            continue
        key = lc.split(":")[0].strip().upper()
        val = lc.split(":", 1)[1].strip()

        if key == "CLASSIFICATION":
            for cls in ["glioma", "meningioma", "pituitary", "healthy"]:
                if cls in val.lower():
                    classification = cls
                    break
        elif key == "CONFIDENCE":
            for conf in ["high", "medium", "low"]:
                if conf in val.lower():
                    confidence = conf
                    break
        elif key == "REASONING":
            r_lines = [val]
            j = idx + 1
            while j < len(lines):
                nk = lines[j].strip().split(":")[0].strip().upper()
                if nk in ("CLASSIFICATION","CONFIDENCE",
                          "STEP1","STEP2","STEP3","STEP4"):
                    break
                if lines[j].strip():
                    r_lines.append(lines[j].strip())
                j += 1
            reasoning = " ".join(r_lines)
            idx = j
            continue
        elif key.startswith("STEP"):
            steps[key] = val
        idx += 1

    return {
        "classification": classification,
        "confidence":     confidence,
        "reasoning":      reasoning,
        "steps":          steps,
        "raw_response":   raw,
    }


# ══════════════════════════════════════════════════════════════════
#  PATIENT CLASSIFIER
# ══════════════════════════════════════════════════════════════════

def classify_patient(path: str, model, tokenizer) -> dict:
    import time
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    nifti_path, modality_type = find_best_modality_file(path)
    if nifti_path is None:
        print(f"  WARNING: No NIfTI found in {path}")
        return {
            "patient": patient_name, "nifti_file": None,
            "modality_used": None, "classification": "ERROR",
            "confidence": "N/A", "reasoning": "No NIfTI file found.",
            "steps": {}, "raw_response": "",
        }

    print(f"    File     : {Path(nifti_path).name}  [{modality_type}]")

    # Preprocess
    t0 = time.time()
    try:
        vol = preprocess_volume(nifti_path)
        print(f"    Preproc  : {tuple(vol.shape)}  [{time.time()-t0:.1f}s]")
    except Exception as e:
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Preprocessing failed: {e}",
            "steps": {}, "raw_response": "",
        }

    # Inference
    t1 = time.time()
    try:
        raw = run_inference(vol, model, tokenizer)
        print(f"    Inference: {time.time()-t1:.1f}s")
    except Exception as e:
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Inference failed: {e}",
            "steps": {}, "raw_response": "",
        }

    parsed = parse_response(raw)
    return {
        "patient":       patient_name,
        "nifti_file":    nifti_path,
        "modality_used": modality_type,
        **parsed,
    }


# ══════════════════════════════════════════════════════════════════
#  DATASET PATH HELPERS
# ══════════════════════════════════════════════════════════════════

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    base = Path(dataset_dir)
    if dataset_type == "glioma":
        paths = []
        for grade in ["HGG", "LGG"]:
            gd = base / grade
            if gd.exists():
                paths += [str(d) for d in sorted(gd.iterdir()) if d.is_dir()]
        return paths or [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "pituitary":
        paths = []
        for sub in sorted(base.iterdir()):
            if sub.is_dir() and sub.name.startswith("sub-"):
                anat = sub / "anat"
                paths.append(str(anat) if anat.exists() else str(sub))
        return paths
    elif dataset_type == "healthy":
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])
    else:
        raise ValueError(f"Unknown dataset_type: '{dataset_type}'")


# ══════════════════════════════════════════════════════════════════
#  BATCH RUNNER
# ══════════════════════════════════════════════════════════════════

def run_batch(dataset_dir: str, dataset_type: str, true_label: str,
              output_csv: str = "results.csv",
              max_patients: int = None) -> list:

    model, tokenizer = load_model()
    patient_paths    = get_patient_paths(dataset_dir, dataset_type)
    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*60}")
    print(f"  BATCH       : {dataset_type.upper()}")
    print(f"  Patients    : {len(patient_paths)}")
    print(f"  True label  : {true_label}")
    print(f"  Output      : {output_csv}")
    print(f"{'='*60}\n")

    results = []
    correct = 0

    for i, path in enumerate(tqdm(patient_paths, desc=dataset_type)):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        print(f"\n  [{i+1}/{len(patient_paths)}] {name}")

        result               = classify_patient(path, model, tokenizer)
        result["true_label"] = true_label
        result["correct"]    = result["classification"] == true_label
        if result["correct"]:
            correct += 1

        tick = "✓ CORRECT" if result["correct"] else f"✗ WRONG → {result['classification']}"
        print(f"    Result   : {result['classification'].upper()} [{result['confidence']}]  {tick}")

        for k, v in result.get("steps", {}).items():
            result[k.lower()] = v
        results.append(result)

    fieldnames = ["patient", "true_label", "classification", "confidence",
                  "correct", "modality_used", "step1", "step2", "step3",
                  "step4", "reasoning", "nifti_file"]
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    acc = correct / len(results) * 100 if results else 0
    print(f"\n{'='*60}")
    print(f"  Accuracy    : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Predictions : {dict(Counter(r['classification'] for r in results))}")
    print(f"  Saved to    : {output_csv}")
    print(f"{'='*60}\n")
    return results


# ══════════════════════════════════════════════════════════════════
#  SINGLE PATIENT RUNNER
# ══════════════════════════════════════════════════════════════════

def run_single(path: str, true_label: str = None) -> dict:
    model, tokenizer = load_model()
    name = Path(path).stem if Path(path).is_file() else Path(path).name

    print(f"\n{'='*60}")
    print(f"  Patient     : {name}")
    print(f"  Model       : Med3DVLM (3D volumetric)")
    print(f"{'='*60}")

    result = classify_patient(path, model, tokenizer)

    print(f"\n  Modality       : {result.get('modality_used', 'N/A')}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")
    if true_label:
        match = "✓ CORRECT" if result["classification"] == true_label else "✗ WRONG"
        print(f"  True label     : {true_label}  →  {match}")

    steps = result.get("steps", {})
    if steps:
        print(f"\n  Step-by-step analysis:")
        for k in sorted(steps.keys()):
            print(f"    {k}: {steps[k]}")

    print(f"\n  Reasoning : {result['reasoning']}")
    print(f"\n  Full response:")
    print(f"  {'-'*55}")
    for line in result["raw_response"].splitlines():
        print(f"  {line}")
    print(f"{'='*60}\n")
    return result


# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE AND RUN
# ══════════════════════════════════════════════════════════════════

RUN_MODE = "single"   # "single" or "batch"

# Single patient
SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000"
SINGLE_LABEL = "meningioma"

# Batch
BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training"
BATCH_TYPE   = "glioma"
BATCH_LABEL  = "glioma"
BATCH_MAX    = 10
BATCH_OUTPUT = f"{BATCH_TYPE}_med3dvlm_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

if __name__ == "__main__":
    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)
    elif RUN_MODE == "batch":
        run_batch(BATCH_DIR, BATCH_TYPE, BATCH_LABEL,
                  output_csv=BATCH_OUTPUT, max_patients=BATCH_MAX)

  Repo path : /home/23bq1a4293/Personal/project/Med3DVLM added to sys.path

  Loading Med3DVLM
  Weights    : /home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM
  modeling.py: FOUND ✓
  Device     : cuda:0
  Added <image> token to tokenizer


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

  VRAM     : 15.4GB used | 32.3GB free
  Model ready ✓

  Patient     : BraTS-MEN-00154-000
  Model       : Med3DVLM (3D volumetric)
    File     : BraTS-MEN-00154-000-t1c.nii.gz  [t1c_brats2024]
    Preproc  : (1, 1, 32, 256, 256)  [0.5s]

  Modality       : t1c_brats2024
  CLASSIFICATION : ERROR
  CONFIDENCE     : N/A
  True label     : meningioma  →  ✗ WRONG

  Reasoning : Inference failed: 'NoneType' object has no attribute 'shape'

  Full response:
  -------------------------------------------------------



In [2]:
# Run this cell in your notebook to patch the inference function

def run_inference_fixed(image_tensor, model, processor, proj_out_num):
    """Exact replica of app.py generate block."""
    import torch
    
    DEVICE = image_tensor.device
    MAX_LENGTH = 512
    
    # Build prompt — exact app.py pattern
    image_tokens = "<im_patch>" * proj_out_num
    input_txt    = image_tokens + QUESTION

    # Tokenize — app.py uses processor(text)["input_ids"]
    input_ids = processor(
        input_txt,
        return_tensors="pt"
    )["input_ids"].to(device=DEVICE)

    print(f"    input_ids shape : {input_ids.shape}")
    print(f"    image shape     : {image_tensor.shape}")
    print(f"    proj_out_num    : {proj_out_num}")

    with torch.no_grad():
        generation = model.generate(
            images=image_tensor,    # key 1: images=
            inputs=input_ids,       # key 2: inputs= (NOT input_ids=)
            max_new_tokens=MAX_LENGTH,
            do_sample=True,         # app.py uses do_sample=True
            top_p=0.9,              # app.py uses top_p=0.9
            temperature=1.0,        # app.py uses temperature=1.0
        )

    print(f"    generation type : {type(generation)}")
    print(f"    generation      : {generation}")

    if generation is None:
        return "ERROR: model.generate() returned None"

    result = processor.decode(generation[0], skip_special_tokens=True)
    return result.strip()


# Also fix the image size — app.py uses (128, 256, 256) not (32, 256, 256)
def load_and_resize_fixed(nifti_path):
    import SimpleITK as sitk
    import numpy as np
    import torch
    from monai.transforms import Resize
    
    DEVICE = "cuda:0"
    DTYPE  = torch.bfloat16
    IMAGE_SIZE = (128, 256, 256)   # MUST match app.py exactly
    
    img       = sitk.ReadImage(str(nifti_path))
    img_array = sitk.GetArrayFromImage(img).astype(np.float32)
    
    image_input = np.expand_dims(img_array.copy(), axis=0)  # [1, D, H, W]
    
    resize_transform = Resize(spatial_size=IMAGE_SIZE, mode="bilinear")
    image_input      = resize_transform(image_input)         # [1, 128, 256, 256]
    
    image_tensor = image_input.data.unsqueeze(0).to(device=DEVICE, dtype=DTYPE)
    # shape: [1, 1, 128, 256, 256]
    
    print(f"    Image tensor shape: {image_tensor.shape}")
    return image_tensor


# Test it directly
nifti_path = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000/BraTS-MEN-00154-000-t1c.nii.gz"

image_tensor = load_and_resize_fixed(nifti_path)
raw = run_inference_fixed(image_tensor, _model, _processor, _proj_out_num)
print("\nRaw output:")
print(raw)

    Image tensor shape: torch.Size([1, 1, 128, 256, 256])


NameError: name '_processor' is not defined

In [3]:
import os, re, sys, torch, numpy as np, SimpleITK as sitk
from pathlib import Path
from monai.transforms import Resize
from transformers import AutoProcessor, AutoModelForCausalLM

# ── Paths ──────────────────────────────────────────────────────
WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"
REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
DEVICE  = "cuda:0"
DTYPE   = torch.bfloat16

# ── Add repo to path ───────────────────────────────────────────
for p in [REPO, str(Path(REPO) / "src")]:
    if Path(p).exists() and p not in sys.path:
        sys.path.insert(0, p)

# ── Load model ─────────────────────────────────────────────────
print("Loading processor...")
processor = AutoProcessor.from_pretrained(
    WEIGHTS, max_length=512, padding_side="right",
    use_fast=False, trust_remote_code=True, ignore_mismatched_sizes=True,
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    WEIGHTS, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True,
).to(device=DEVICE).eval()

try:
    proj_out_num = model.get_model().config.proj_out_num
except:
    proj_out_num = 256
print(f"proj_out_num = {proj_out_num}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f}GB")

Loading processor...
Loading model...


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

proj_out_num = 256
VRAM used: 30.7GB


In [4]:
# ── Load and resize volume ─────────────────────────────────────
nifti_path = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000/BraTS-MEN-00154-000-t1c.nii.gz"

img       = sitk.ReadImage(nifti_path)
img_array = sitk.GetArrayFromImage(img).astype(np.float32)
print(f"Raw array shape: {img_array.shape}")

image_input = np.expand_dims(img_array.copy(), axis=0)
resize_fn   = Resize(spatial_size=(128, 256, 256), mode="bilinear")
image_input = resize_fn(image_input)
image_tensor = image_input.data.unsqueeze(0).to(device=DEVICE, dtype=DTYPE)
print(f"Image tensor shape: {image_tensor.shape}")

Raw array shape: (155, 240, 240)
Image tensor shape: torch.Size([1, 1, 128, 256, 256])


In [5]:
# ── Run inference ──────────────────────────────────────────────
QUESTION = (
    "You are a neuroradiologist. Classify this 3D brain MRI into exactly one of: "
    "glioma | meningioma | pituitary | healthy\n"
    "Respond as:\n"
    "CLASSIFICATION: <class>\n"
    "CONFIDENCE: <high|medium|low>\n"
    "REASONING: <why>"
)

image_tokens = "<im_patch>" * proj_out_num
input_txt    = image_tokens + QUESTION
input_ids    = processor(input_txt, return_tensors="pt")["input_ids"].to(DEVICE)

print(f"input_ids shape : {input_ids.shape}")
print(f"image shape     : {image_tensor.shape}")

with torch.no_grad():
    generation = model.generate(
        images=image_tensor,
        inputs=input_ids,
        max_new_tokens=200,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
    )

print(f"generation type : {type(generation)}")
print(f"generation      : {generation}")

if generation is not None:
    result = processor.decode(generation[0], skip_special_tokens=True)
    print(f"\nResult:\n{result}")

input_ids shape : torch.Size([1, 317])
image shape     : torch.Size([1, 1, 128, 256, 256])


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


generation type : <class 'monai.data.meta_tensor.MetaTensor'>
generation      : tensor([[151645]], device='cuda:0')

Result:



In [6]:
# Run this to see the exact generate() signature from modeling.py
import inspect

# Find the generate method in the model
for name, obj in inspect.getmembers(model, predicate=inspect.ismethod):
    if name == 'generate':
        print(inspect.signature(obj))
        break

# Also check the source
try:
    src = inspect.getsource(model.generate)
    print(src[:3000])
except:
    # Check parent class
    for cls in type(model).__mro__:
        if 'generate' in cls.__dict__:
            print(f"generate defined in: {cls}")
            try:
                src = inspect.getsource(cls.__dict__['generate'])
                print(src[:3000])
            except:
                pass
            break

(images: Optional[torch.Tensor] = None, inputs: Optional[torch.Tensor] = None, **kwargs) -> Union[transformers.generation.utils.GenerateDecoderOnlyOutput, transformers.generation.utils.GenerateEncoderDecoderOutput, transformers.generation.utils.GenerateBeamDecoderOnlyOutput, transformers.generation.utils.GenerateBeamEncoderDecoderOutput, torch.LongTensor, Any]
    @torch.no_grad()
    def generate(
        self,
        images: Optional[torch.Tensor] = None,
        inputs: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> Union[GenerateOutput, torch.LongTensor, Any]:
        position_ids = kwargs.pop("position_ids", None)
        attention_mask = kwargs.pop("attention_mask", None)
        if "inputs_embeds" in kwargs:
            raise NotImplementedError("`inputs_embeds` is not supported")

        if images is not None:
            (inputs, position_ids, attention_mask, _, inputs_embeds, _) = (
                self.prepare_inputs_for_multimodal(
                    inputs,
 

In [7]:
"""
Med3DVLM Brain Tumor Classification — Final Correct Pipeline
=============================================================
Built from exact modeling.py analysis:

  prepare_inputs_for_multimodal(input_ids, position_ids, attention_mask,
                                 past_key_values, labels, images)
  → returns (None, position_ids, attention_mask, past_key_values,
              inputs_embeds, labels)

  The correct inference flow:
    1. Tokenize: image_tokens + question → input_ids
    2. Call prepare_inputs_for_multimodal → inputs_embeds
       (this replaces <im_patch> tokens with actual 3D image features)
    3. Call model.generate(inputs_embeds=...) — text generation
       from the fused image+text embeddings

Paths (update before running):
  MED3DVLM_REPO    : cloned github repo (has src/)
  MED3DVLM_WEIGHTS : downloaded MagicXin weights folder

Requirements:
  pip install SimpleITK monai transformers torch nibabel scipy tqdm
"""

import os
import re
import csv
import sys
import warnings
import numpy as np
import nibabel as nib
import SimpleITK as sitk

from pathlib import Path
from collections import Counter
from datetime import datetime
from tqdm import tqdm

import torch
from monai.transforms import Resize
from transformers import AutoProcessor, AutoModelForCausalLM

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE PATHS
# ══════════════════════════════════════════════════════════════════

MED3DVLM_REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
MED3DVLM_WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"

DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16
MAX_LENGTH = 512
IMAGE_SIZE = (128, 256, 256)   # exact value from app.py

# ══════════════════════════════════════════════════════════════════
#  ADD REPO SRC TO PATH
# ══════════════════════════════════════════════════════════════════

def setup_pythonpath():
    repo = Path(MED3DVLM_REPO)
    if repo.exists():
        for p in [str(repo), str(repo / "src")]:
            if p not in sys.path:
                sys.path.insert(0, p)

setup_pythonpath()

# ══════════════════════════════════════════════════════════════════
#  MODEL LOADER
# ══════════════════════════════════════════════════════════════════

_model        = None
_processor    = None
_proj_out_num = 256

def load_model():
    global _model, _processor, _proj_out_num

    if _model is not None:
        print("  Model already loaded — reusing")
        return _model, _processor, _proj_out_num

    weights = Path(MED3DVLM_WEIGHTS).resolve()
    if not weights.exists():
        raise FileNotFoundError(
            f"Weights not found at: {weights}\n"
            "Download: from huggingface_hub import snapshot_download\n"
            "snapshot_download(repo_id='MagicXin/Med3DVLM-Qwen-2.5-7B', "
            f"local_dir='{weights}')"
        )

    print(f"\n{'='*60}")
    print(f"  Loading Med3DVLM")
    print(f"  Weights : {weights}")
    print(f"  Device  : {DEVICE}")
    print(f"{'='*60}")

    _processor = AutoProcessor.from_pretrained(
        str(weights),
        max_length=MAX_LENGTH,
        padding_side="right",
        use_fast=False,
        trust_remote_code=True,
        ignore_mismatched_sizes=True,
    )

    _model = AutoModelForCausalLM.from_pretrained(
        str(weights),
        torch_dtype=DTYPE,
        device_map=DEVICE,
        trust_remote_code=True,
    ).to(device=DEVICE).eval()

    # Get proj_out_num from model config
    try:
        _proj_out_num = _model.get_model().config.proj_out_num
        print(f"  proj_out_num : {_proj_out_num}")
    except Exception:
        _proj_out_num = 256
        print(f"  proj_out_num : {_proj_out_num} (default)")

    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  VRAM     : {used:.1f}GB used | {total - used:.1f}GB free")

    print("  Model ready ✓\n")
    return _model, _processor, _proj_out_num


# ══════════════════════════════════════════════════════════════════
#  MODALITY FILE FINDER
# ══════════════════════════════════════════════════════════════════

MODALITY_PATTERNS = {
    "t1ce_brats":        re.compile(r".*t1ce\.nii(\.gz)?$",                    re.IGNORECASE),
    "t1c_brats2024":     re.compile(r".*-t1c\.nii(\.gz)?$",                    re.IGNORECASE),
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$",  re.IGNORECASE),
    "healthy":           re.compile(r"^healthy_\d+\.nii(\.gz)?$",              re.IGNORECASE),
}
PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024",
    "pituitary_cecor", "pituitary_cesag", "pituitary_ce3dnav", "healthy",
]
SKIP_KEYWORDS = [
    "seg", "mask", "label", "defacemask", "seed",
    "prediction", "groundtruth", "dwi", "adcb"
]

def find_best_modality_file(path: str) -> tuple:
    p = Path(path)
    if p.is_file():
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(p.name):
                return str(p), key
        if p.name.endswith(".nii") or p.name.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None
    if not p.is_dir():
        return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates
                  if not any(kw in f.name.lower() for kw in SKIP_KEYWORDS)]
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches:
            return str(matches[0]), key
    return (str(candidates[0]), "fallback") if candidates else (None, None)


# ══════════════════════════════════════════════════════════════════
#  IMAGE PREPROCESSING  — exact app.py pattern
# ══════════════════════════════════════════════════════════════════

def load_and_resize_volume(nifti_path: str) -> torch.Tensor:
    """
    Exact replica of app.py preprocessing:
        img = sitk.ReadImage(file.name)
        img_array = sitk.GetArrayFromImage(img)
        image_input = np.expand_dims(img_array.copy(), axis=0)
        image_input = Resize(spatial_size=(128,256,256), mode="bilinear")(image_input)
        image_input = image_input.data.unsqueeze(0).to(device=device, dtype=dtype)
    Returns: [1, 1, 128, 256, 256] bfloat16 tensor on DEVICE
    """
    img          = sitk.ReadImage(str(nifti_path))
    img_array    = sitk.GetArrayFromImage(img).astype(np.float32)
    image_input  = np.expand_dims(img_array.copy(), axis=0)        # [1, D, H, W]

    resize_fn    = Resize(spatial_size=IMAGE_SIZE, mode="bilinear")
    image_input  = resize_fn(image_input)                           # [1, 128, 256, 256]

    image_tensor = image_input.data.unsqueeze(0).to(                # [1, 1, 128, 256, 256]
        device=DEVICE, dtype=DTYPE
    )
    return image_tensor


# ══════════════════════════════════════════════════════════════════
#  CLASSIFICATION PROMPT
# ══════════════════════════════════════════════════════════════════

QUESTION = (
    "You are a consultant neuroradiologist analyzing a full 3D brain MRI volume "
    "(T1 contrast-enhanced). "
    "Classify into EXACTLY ONE of: glioma | meningioma | pituitary | healthy\n\n"

    "RULE 1 — APPLY FIRST: Does the enhancing mass TOUCH the inner skull surface?\n"
    "  YES → meningioma (extra-axial, dura-based)\n"
    "  NO  → check if inside brain (glioma) or at midline skull base (pituitary)\n\n"

    "GLIOMA: Inside brain parenchyma. Fuzzy infiltrative borders. "
    "Ring enhancement + dark necrotic center (HGG) or absent enhancement (LGG).\n"
    "MENINGIOMA: Outside brain on dura/skull. Sharp clean border. "
    "Solid homogeneous bright enhancement. Dural tail sign.\n"
    "PITUITARY: Midline skull base in sella only. "
    "Snowman shape (macro) or dark spot in bright gland (micro).\n"
    "HEALTHY: No focal mass. Normal symmetric brain. "
    "Normal pituitary enhancement is NOT a tumor.\n\n"

    "Respond in EXACTLY this format, no other text:\n"
    "STEP1: <skull contact test result>\n"
    "STEP2: <location — intra-axial / extra-axial / sellar / none>\n"
    "STEP3: <borders — sharp or infiltrative>\n"
    "STEP4: <enhancement — homogeneous solid / ring+necrosis / minimal / absent>\n"
    "CLASSIFICATION: <glioma|meningioma|pituitary|healthy>\n"
    "CONFIDENCE: <high|medium|low>\n"
    "REASONING: <3 sentences: cite 3D spatial features, rule out runner-up>"
)


# ══════════════════════════════════════════════════════════════════
#  INFERENCE — correct pattern from modeling.py
#
#  prepare_inputs_for_multimodal signature:
#    (self, input_ids, position_ids, attention_mask,
#     past_key_values, labels, images)
#  → returns (None, pos_ids, attn_mask, pkv, inputs_embeds, labels)
#
#  Correct flow:
#    1. Build input_ids with <im_patch> tokens prepended
#    2. Call prepare_inputs_for_multimodal → get inputs_embeds
#       (image features injected into embedding space)
#    3. Call model.generate(inputs_embeds=inputs_embeds)
# ══════════════════════════════════════════════════════════════════

def run_inference(image_tensor: torch.Tensor,
                  model, processor, proj_out_num: int) -> str:
    """
    Correct Med3DVLM inference using prepare_inputs_for_multimodal.
    """
    # ── Step 1: Build tokenized input ────────────────────────────
    image_tokens   = "<im_patch>" * proj_out_num
    input_txt      = image_tokens + QUESTION
    input_ids      = processor(
        input_txt, return_tensors="pt"
    )["input_ids"].to(device=DEVICE)

    attention_mask = torch.ones_like(input_ids)
    position_ids   = torch.arange(
        input_ids.shape[1], device=DEVICE
    ).unsqueeze(0)

    # ── Step 2: Fuse image features into embeddings ───────────────
    with torch.no_grad():
        (
            _,              # input_ids → None (replaced by embeds)
            _,              # position_ids
            attention_mask_mm,
            _,              # past_key_values
            inputs_embeds,  # ← fused image+text embeddings
            _,              # labels
        ) = model.prepare_inputs_for_multimodal(
            input_ids      = input_ids,
            position_ids   = position_ids,
            attention_mask = attention_mask,
            past_key_values= None,
            labels         = None,
            images         = image_tensor,
        )

    # ── Step 3: Generate from embeddings ─────────────────────────
    # Get eos token id safely
    try:
        eos_id = processor.tokenizer.eos_token_id
    except AttributeError:
        eos_id = processor.eos_token_id

    with torch.no_grad():
        generation = model.generate(
            inputs_embeds  = inputs_embeds,
            attention_mask = attention_mask_mm,
            max_new_tokens = 300,
            do_sample      = False,      # greedy — deterministic
            temperature    = None,
            top_p          = None,
            use_cache      = True,
            pad_token_id   = eos_id,
        )

    # Decode
    raw = processor.decode(generation[0], skip_special_tokens=True).strip()

    del image_tensor, input_ids, inputs_embeds, generation
    torch.cuda.empty_cache()

    return raw


# ══════════════════════════════════════════════════════════════════
#  RESPONSE PARSER
# ══════════════════════════════════════════════════════════════════

def parse_response(raw: str) -> dict:
    classification = "UNKNOWN"
    confidence     = "UNKNOWN"
    reasoning      = ""
    steps          = {}

    lines = raw.splitlines()
    idx   = 0
    while idx < len(lines):
        lc = lines[idx].strip()
        if not lc or ":" not in lc:
            idx += 1
            continue
        key = lc.split(":")[0].strip().upper()
        val = lc.split(":", 1)[1].strip()

        if key == "CLASSIFICATION":
            for cls in ["glioma", "meningioma", "pituitary", "healthy"]:
                if cls in val.lower():
                    classification = cls
                    break
        elif key == "CONFIDENCE":
            for conf in ["high", "medium", "low"]:
                if conf in val.lower():
                    confidence = conf
                    break
        elif key == "REASONING":
            r_lines = [val]
            j = idx + 1
            while j < len(lines):
                nk = lines[j].strip().split(":")[0].strip().upper()
                if nk in ("CLASSIFICATION", "CONFIDENCE",
                          "STEP1", "STEP2", "STEP3", "STEP4"):
                    break
                if lines[j].strip():
                    r_lines.append(lines[j].strip())
                j += 1
            reasoning = " ".join(r_lines)
            idx = j
            continue
        elif key.startswith("STEP"):
            steps[key] = val
        idx += 1

    return {
        "classification": classification,
        "confidence":     confidence,
        "reasoning":      reasoning,
        "steps":          steps,
        "raw_response":   raw,
    }


# ══════════════════════════════════════════════════════════════════
#  PATIENT CLASSIFIER
# ══════════════════════════════════════════════════════════════════

def classify_patient(path: str, model, processor, proj_out_num: int) -> dict:
    import time
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    nifti_path, modality_type = find_best_modality_file(path)
    if nifti_path is None:
        print(f"  WARNING: No NIfTI found in {path}")
        return {
            "patient": patient_name, "nifti_file": None,
            "modality_used": None, "classification": "ERROR",
            "confidence": "N/A", "reasoning": "No NIfTI file found.",
            "steps": {}, "raw_response": "",
        }

    print(f"    File     : {Path(nifti_path).name}  [{modality_type}]")

    # Load and resize
    t0 = time.time()
    try:
        image_tensor = load_and_resize_volume(nifti_path)
        print(f"    Volume   : {tuple(image_tensor.shape)}  [{time.time()-t0:.1f}s]")
    except Exception as e:
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Volume loading failed: {e}",
            "steps": {}, "raw_response": "",
        }

    # Inference
    t1 = time.time()
    try:
        raw = run_inference(image_tensor, model, processor, proj_out_num)
        print(f"    Inference: {time.time()-t1:.1f}s")
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Inference failed: {e}",
            "steps": {}, "raw_response": "",
        }

    parsed = parse_response(raw)
    return {
        "patient":       patient_name,
        "nifti_file":    nifti_path,
        "modality_used": modality_type,
        **parsed,
    }


# ══════════════════════════════════════════════════════════════════
#  DATASET PATH HELPERS
# ══════════════════════════════════════════════════════════════════

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    base = Path(dataset_dir)
    if dataset_type == "glioma":
        paths = []
        for grade in ["HGG", "LGG"]:
            gd = base / grade
            if gd.exists():
                paths += [str(d) for d in sorted(gd.iterdir()) if d.is_dir()]
        return paths or [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "pituitary":
        paths = []
        for sub in sorted(base.iterdir()):
            if sub.is_dir() and sub.name.startswith("sub-"):
                anat = sub / "anat"
                paths.append(str(anat) if anat.exists() else str(sub))
        return paths
    elif dataset_type == "healthy":
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])
    else:
        raise ValueError(f"Unknown dataset_type: '{dataset_type}'")


# ══════════════════════════════════════════════════════════════════
#  BATCH RUNNER
# ══════════════════════════════════════════════════════════════════

def run_batch(dataset_dir: str, dataset_type: str, true_label: str,
              output_csv: str = "results.csv",
              max_patients: int = None) -> list:

    model, processor, proj_out_num = load_model()
    patient_paths = get_patient_paths(dataset_dir, dataset_type)
    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*60}")
    print(f"  BATCH       : {dataset_type.upper()}")
    print(f"  Patients    : {len(patient_paths)}")
    print(f"  True label  : {true_label}")
    print(f"  Output      : {output_csv}")
    print(f"{'='*60}\n")

    results = []
    correct = 0

    for i, path in enumerate(tqdm(patient_paths, desc=dataset_type)):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        print(f"\n  [{i+1}/{len(patient_paths)}] {name}")

        result               = classify_patient(path, model, processor, proj_out_num)
        result["true_label"] = true_label
        result["correct"]    = result["classification"] == true_label
        if result["correct"]:
            correct += 1

        tick = "✓ CORRECT" if result["correct"] else f"✗ WRONG → {result['classification']}"
        print(f"    Result   : {result['classification'].upper()} [{result['confidence']}]  {tick}")

        for k, v in result.get("steps", {}).items():
            result[k.lower()] = v
        results.append(result)

    fieldnames = ["patient", "true_label", "classification", "confidence",
                  "correct", "modality_used", "step1", "step2", "step3",
                  "step4", "reasoning", "nifti_file"]
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    acc = correct / len(results) * 100 if results else 0
    print(f"\n{'='*60}")
    print(f"  Accuracy    : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Predictions : {dict(Counter(r['classification'] for r in results))}")
    print(f"  Saved to    : {output_csv}")
    print(f"{'='*60}\n")
    return results


# ══════════════════════════════════════════════════════════════════
#  SINGLE PATIENT RUNNER
# ══════════════════════════════════════════════════════════════════

def run_single(path: str, true_label: str = None) -> dict:
    model, processor, proj_out_num = load_model()
    name = Path(path).stem if Path(path).is_file() else Path(path).name

    print(f"\n{'='*60}")
    print(f"  Patient      : {name}")
    print(f"  Model        : Med3DVLM  proj_out_num={proj_out_num}")
    print(f"{'='*60}")

    result = classify_patient(path, model, processor, proj_out_num)

    print(f"\n  Modality       : {result.get('modality_used', 'N/A')}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")
    if true_label:
        match = "✓ CORRECT" if result["classification"] == true_label else "✗ WRONG"
        print(f"  True label     : {true_label}  →  {match}")

    steps = result.get("steps", {})
    if steps:
        print(f"\n  Step-by-step:")
        for k in sorted(steps.keys()):
            print(f"    {k}: {steps[k]}")

    print(f"\n  Reasoning : {result['reasoning']}")
    print(f"\n  Full response:")
    print(f"  {'-'*55}")
    for line in result["raw_response"].splitlines():
        print(f"  {line}")
    print(f"{'='*60}\n")
    return result


# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE AND RUN
# ══════════════════════════════════════════════════════════════════

RUN_MODE = "single"   # "single" or "batch"

SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000"
SINGLE_LABEL = "meningioma"

BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training"
BATCH_TYPE   = "glioma"
BATCH_LABEL  = "glioma"
BATCH_MAX    = 10
BATCH_OUTPUT = f"{BATCH_TYPE}_med3dvlm_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

if __name__ == "__main__":
    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)
    elif RUN_MODE == "batch":
        run_batch(BATCH_DIR, BATCH_TYPE, BATCH_LABEL,
                  output_csv=BATCH_OUTPUT, max_patients=BATCH_MAX)


  Loading Med3DVLM
  Weights : /home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM
  Device  : cuda:0


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

  proj_out_num : 256
  VRAM     : 30.8GB used | 16.9GB free
  Model ready ✓


  Patient      : BraTS-MEN-00154-000
  Model        : Med3DVLM  proj_out_num=256
    File     : BraTS-MEN-00154-000-t1c.nii.gz  [t1c_brats2024]
    Volume   : (1, 1, 128, 256, 256)  [0.2s]

  Modality       : t1c_brats2024
  CLASSIFICATION : ERROR
  CONFIDENCE     : N/A
  True label     : meningioma  →  ✗ WRONG

  Reasoning : Inference failed: `inputs_embeds` is not supported

  Full response:
  -------------------------------------------------------



Traceback (most recent call last):
  File "/tmp/ipykernel_258198/3817472.py", line 408, in classify_patient
    raw = run_inference(image_tensor, model, processor, proj_out_num)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_258198/3817472.py", line 295, in run_inference
    generation = model.generate(
                 ^^^^^^^^^^^^^^^
  File "/home/23bq1a4293/.local/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/23bq1a4293/.cache/huggingface/modules/transformers_modules/Med3DVLM/modeling.py", line 1686, in generate
    raise NotImplementedError("`inputs_embeds` is not supported")
NotImplementedError: `inputs_embeds` is not supported


In [8]:
# Inspect the custom generate() to find correct kwargs
import inspect
src = inspect.getsource(model.generate)
print(src[:4000])

    @torch.no_grad()
    def generate(
        self,
        images: Optional[torch.Tensor] = None,
        inputs: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> Union[GenerateOutput, torch.LongTensor, Any]:
        position_ids = kwargs.pop("position_ids", None)
        attention_mask = kwargs.pop("attention_mask", None)
        if "inputs_embeds" in kwargs:
            raise NotImplementedError("`inputs_embeds` is not supported")

        if images is not None:
            (inputs, position_ids, attention_mask, _, inputs_embeds, _) = (
                self.prepare_inputs_for_multimodal(
                    inputs,
                    position_ids,
                    attention_mask,
                    None,
                    None,
                    images,
                )
            )
        else:
            inputs_embeds = self.get_model().embed_tokens(inputs)

        output_ids = super().generate(inputs_embeds=inputs_embeds, **kwargs)
        return output_id

In [9]:
"""
Med3DVLM Brain Tumor Classification — Final Working Pipeline
=============================================================
Based on exact generate() signature from modeling.py:

    def generate(self,
        images: Optional[torch.Tensor] = None,   # [1,1,128,256,256]
        inputs: Optional[torch.Tensor] = None,   # input_ids
        **kwargs
    )
    → internally calls prepare_inputs_for_multimodal then super().generate()

Correct call:
    model.generate(
        images=image_tensor,
        inputs=input_ids,
        max_new_tokens=...,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
    )

DO NOT pass: inputs_embeds, input_ids, attention_mask, position_ids
These are all handled internally by the custom generate().

Datasets:
  Glioma      : *_t1ce.nii          (BraTS 2019)
  Meningioma  : *-t1c.nii.gz        (BraTS 2024)
  Pituitary   : *acq-CECor_T1w*     (BIDS)
  Healthy     : healthy_XXX.nii.gz  (flat)
"""

import os
import re
import csv
import sys
import warnings
import numpy as np
import nibabel as nib
import SimpleITK as sitk

from pathlib import Path
from collections import Counter
from datetime import datetime
from tqdm import tqdm

import torch
from monai.transforms import Resize
from transformers import AutoProcessor, AutoModelForCausalLM

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE PATHS
# ══════════════════════════════════════════════════════════════════

MED3DVLM_REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
MED3DVLM_WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"

DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16
MAX_LENGTH = 512
IMAGE_SIZE = (128, 256, 256)   # exact from app.py

# ══════════════════════════════════════════════════════════════════
#  ADD REPO SRC TO PATH
# ══════════════════════════════════════════════════════════════════

def setup_pythonpath():
    repo = Path(MED3DVLM_REPO)
    if repo.exists():
        for p in [str(repo), str(repo / "src")]:
            if p not in sys.path:
                sys.path.insert(0, p)

setup_pythonpath()

# ══════════════════════════════════════════════════════════════════
#  MODEL LOADER
# ══════════════════════════════════════════════════════════════════

_model        = None
_processor    = None
_proj_out_num = 256

def load_model():
    global _model, _processor, _proj_out_num

    if _model is not None:
        print("  Model already loaded — reusing")
        return _model, _processor, _proj_out_num

    weights = Path(MED3DVLM_WEIGHTS).resolve()
    if not weights.exists():
        raise FileNotFoundError(
            f"Weights not found at: {weights}\n"
            "Download:\n"
            "  from huggingface_hub import snapshot_download\n"
            "  snapshot_download(\n"
            "      repo_id='MagicXin/Med3DVLM-Qwen-2.5-7B',\n"
            f"     local_dir='{weights}'\n"
            "  )"
        )

    print(f"\n{'='*60}")
    print(f"  Loading Med3DVLM")
    print(f"  Weights : {weights}")
    print(f"  Device  : {DEVICE}")
    print(f"{'='*60}")

    # AutoProcessor — exact from app.py
    _processor = AutoProcessor.from_pretrained(
        str(weights),
        max_length=MAX_LENGTH,
        padding_side="right",
        use_fast=False,
        trust_remote_code=True,
        ignore_mismatched_sizes=True,
    )

    # Model
    _model = AutoModelForCausalLM.from_pretrained(
        str(weights),
        torch_dtype=DTYPE,
        device_map=DEVICE,
        trust_remote_code=True,
    ).to(device=DEVICE).eval()

    # proj_out_num — from app.py
    try:
        _proj_out_num = _model.get_model().config.proj_out_num
    except Exception:
        _proj_out_num = 256
    print(f"  proj_out_num : {_proj_out_num}")

    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  VRAM     : {used:.1f}GB used | {total - used:.1f}GB free")

    print("  Model ready ✓\n")
    return _model, _processor, _proj_out_num


# ══════════════════════════════════════════════════════════════════
#  MODALITY FILE FINDER
# ══════════════════════════════════════════════════════════════════

MODALITY_PATTERNS = {
    "t1ce_brats":        re.compile(r".*t1ce\.nii(\.gz)?$",                    re.IGNORECASE),
    "t1c_brats2024":     re.compile(r".*-t1c\.nii(\.gz)?$",                    re.IGNORECASE),
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$",  re.IGNORECASE),
    "healthy":           re.compile(r"^healthy_\d+\.nii(\.gz)?$",              re.IGNORECASE),
}
PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024",
    "pituitary_cecor", "pituitary_cesag", "pituitary_ce3dnav", "healthy",
]
SKIP_KEYWORDS = [
    "seg", "mask", "label", "defacemask", "seed",
    "prediction", "groundtruth", "dwi", "adcb"
]

def find_best_modality_file(path: str) -> tuple:
    p = Path(path)
    if p.is_file():
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(p.name):
                return str(p), key
        if p.name.endswith(".nii") or p.name.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None
    if not p.is_dir():
        return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates
                  if not any(kw in f.name.lower() for kw in SKIP_KEYWORDS)]
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches:
            return str(matches[0]), key
    return (str(candidates[0]), "fallback") if candidates else (None, None)


# ══════════════════════════════════════════════════════════════════
#  IMAGE PREPROCESSING — exact app.py pattern
# ══════════════════════════════════════════════════════════════════

def load_and_resize_volume(nifti_path: str) -> torch.Tensor:
    """
    Exact replica of app.py:
        img = sitk.ReadImage(file.name)
        img_array = sitk.GetArrayFromImage(img)
        image_input = np.expand_dims(img_array.copy(), axis=0)
        image_input = Resize(spatial_size=(128,256,256))(image_input)
        image_input = image_input.data.unsqueeze(0).to(device, dtype)
    Returns: [1, 1, 128, 256, 256] bfloat16 on DEVICE
    """
    img         = sitk.ReadImage(str(nifti_path))
    img_array   = sitk.GetArrayFromImage(img).astype(np.float32)
    image_input = np.expand_dims(img_array.copy(), axis=0)          # [1,D,H,W]
    image_input = Resize(spatial_size=IMAGE_SIZE, mode="bilinear")(image_input)
    image_tensor = image_input.data.unsqueeze(0).to(device=DEVICE, dtype=DTYPE)
    return image_tensor                                              # [1,1,128,256,256]


# ══════════════════════════════════════════════════════════════════
#  CLASSIFICATION PROMPT
# ══════════════════════════════════════════════════════════════════

QUESTION = (
    "You are a consultant neuroradiologist analyzing a full 3D brain MRI "
    "(T1 contrast-enhanced). Classify into EXACTLY ONE of: "
    "glioma | meningioma | pituitary | healthy\n\n"

    "RULE — APPLY FIRST:\n"
    "Does the enhancing mass TOUCH the inner skull surface?\n"
    "  YES → meningioma (extra-axial, dura-based)\n"
    "  NO  → check if inside brain (glioma) or at midline skull base (pituitary)\n\n"

    "GLIOMA: Inside brain parenchyma. Fuzzy infiltrative borders. "
    "Ring enhancement + dark necrotic center (HGG) or absent enhancement (LGG).\n"
    "MENINGIOMA: Outside brain on dura/skull. Sharp border. "
    "Solid homogeneous bright enhancement. Dural tail sign.\n"
    "PITUITARY: Midline skull base in sella only. "
    "Snowman shape (macro) or dark spot in bright gland (micro).\n"
    "HEALTHY: No focal mass. Normal symmetric brain.\n\n"

    "Respond in EXACTLY this format:\n"
    "STEP1: <skull contact test>\n"
    "STEP2: <location — intra-axial/extra-axial/sellar/none>\n"
    "STEP3: <borders — sharp or infiltrative>\n"
    "STEP4: <enhancement — homogeneous/ring+necrosis/minimal/absent>\n"
    "CLASSIFICATION: <glioma|meningioma|pituitary|healthy>\n"
    "CONFIDENCE: <high|medium|low>\n"
    "REASONING: <3 sentences citing 3D features, rule out runner-up>"
)


# ══════════════════════════════════════════════════════════════════
#  INFERENCE — exact generate() signature from modeling.py:
#
#  def generate(self,
#      images: Optional[torch.Tensor] = None,   ← 3D volume
#      inputs: Optional[torch.Tensor] = None,   ← input_ids
#      **kwargs
#  )
#  Internally calls prepare_inputs_for_multimodal then super().generate()
#  DO NOT pass: inputs_embeds, attention_mask, position_ids
# ══════════════════════════════════════════════════════════════════

def run_inference(image_tensor: torch.Tensor,
                  model, processor, proj_out_num: int) -> str:
    """
    Correct inference matching exact generate() signature from modeling.py.
    """
    # Build prompt — <im_patch> × proj_out_num + question (exact app.py)
    image_tokens = "<im_patch>" * proj_out_num
    input_txt    = image_tokens + QUESTION
    input_ids    = processor(
        input_txt, return_tensors="pt"
    )["input_ids"].to(device=DEVICE)

    # Call generate() with exact kwargs from app.py
    # images= and inputs= are the ONLY positional kwargs for this model
    output_ids = model.generate(
        images        = image_tensor,   # [1,1,128,256,256]
        inputs        = input_ids,      # tokenized prompt
        max_new_tokens= MAX_LENGTH,
        do_sample     = True,           # exact app.py values
        top_p         = 0.9,
        temperature   = 1.0,
    )

    # Decode full output then strip the input prompt prefix
    full_result = processor.decode(output_ids[0], skip_special_tokens=True)

    # Remove the image patch tokens and question from the output
    # The model echoes input, so we extract only the answer part
    if "STEP1:" in full_result:
        # Find where the structured response starts
        full_result = full_result[full_result.find("STEP1:"):]
    elif "CLASSIFICATION:" in full_result:
        full_result = full_result[full_result.find("CLASSIFICATION:"):]

    del image_tensor, input_ids, output_ids
    torch.cuda.empty_cache()

    return full_result.strip()


# ══════════════════════════════════════════════════════════════════
#  RESPONSE PARSER
# ══════════════════════════════════════════════════════════════════

def parse_response(raw: str) -> dict:
    classification = "UNKNOWN"
    confidence     = "UNKNOWN"
    reasoning      = ""
    steps          = {}

    lines = raw.splitlines()
    idx   = 0
    while idx < len(lines):
        lc = lines[idx].strip()
        if not lc or ":" not in lc:
            idx += 1
            continue
        key = lc.split(":")[0].strip().upper()
        val = lc.split(":", 1)[1].strip()

        if key == "CLASSIFICATION":
            for cls in ["glioma", "meningioma", "pituitary", "healthy"]:
                if cls in val.lower():
                    classification = cls
                    break
        elif key == "CONFIDENCE":
            for conf in ["high", "medium", "low"]:
                if conf in val.lower():
                    confidence = conf
                    break
        elif key == "REASONING":
            r_lines = [val]
            j = idx + 1
            while j < len(lines):
                nk = lines[j].strip().split(":")[0].strip().upper()
                if nk in ("CLASSIFICATION", "CONFIDENCE",
                          "STEP1", "STEP2", "STEP3", "STEP4"):
                    break
                if lines[j].strip():
                    r_lines.append(lines[j].strip())
                j += 1
            reasoning = " ".join(r_lines)
            idx = j
            continue
        elif key.startswith("STEP"):
            steps[key] = val
        idx += 1

    return {
        "classification": classification,
        "confidence":     confidence,
        "reasoning":      reasoning,
        "steps":          steps,
        "raw_response":   raw,
    }


# ══════════════════════════════════════════════════════════════════
#  PATIENT CLASSIFIER
# ══════════════════════════════════════════════════════════════════

def classify_patient(path: str, model, processor, proj_out_num: int) -> dict:
    import time
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    nifti_path, modality_type = find_best_modality_file(path)
    if nifti_path is None:
        print(f"  WARNING: No NIfTI found in {path}")
        return {
            "patient": patient_name, "nifti_file": None,
            "modality_used": None, "classification": "ERROR",
            "confidence": "N/A", "reasoning": "No NIfTI file found.",
            "steps": {}, "raw_response": "",
        }

    print(f"    File     : {Path(nifti_path).name}  [{modality_type}]")

    # Preprocess
    t0 = time.time()
    try:
        image_tensor = load_and_resize_volume(nifti_path)
        print(f"    Volume   : {tuple(image_tensor.shape)}  [{time.time()-t0:.1f}s]")
    except Exception as e:
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Volume loading failed: {e}",
            "steps": {}, "raw_response": "",
        }

    # Inference
    t1 = time.time()
    try:
        raw = run_inference(image_tensor, model, processor, proj_out_num)
        print(f"    Inference: {time.time()-t1:.1f}s")
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Inference failed: {e}",
            "steps": {}, "raw_response": "",
        }

    parsed = parse_response(raw)
    return {
        "patient":       patient_name,
        "nifti_file":    nifti_path,
        "modality_used": modality_type,
        **parsed,
    }


# ══════════════════════════════════════════════════════════════════
#  DATASET PATH HELPERS
# ══════════════════════════════════════════════════════════════════

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    base = Path(dataset_dir)
    if dataset_type == "glioma":
        paths = []
        for grade in ["HGG", "LGG"]:
            gd = base / grade
            if gd.exists():
                paths += [str(d) for d in sorted(gd.iterdir()) if d.is_dir()]
        return paths or [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "pituitary":
        paths = []
        for sub in sorted(base.iterdir()):
            if sub.is_dir() and sub.name.startswith("sub-"):
                anat = sub / "anat"
                paths.append(str(anat) if anat.exists() else str(sub))
        return paths
    elif dataset_type == "healthy":
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])
    else:
        raise ValueError(f"Unknown dataset_type: '{dataset_type}'")


# ══════════════════════════════════════════════════════════════════
#  BATCH RUNNER
# ══════════════════════════════════════════════════════════════════

def run_batch(dataset_dir: str, dataset_type: str, true_label: str,
              output_csv: str = "results.csv",
              max_patients: int = None) -> list:

    model, processor, proj_out_num = load_model()
    patient_paths = get_patient_paths(dataset_dir, dataset_type)
    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*60}")
    print(f"  BATCH       : {dataset_type.upper()}")
    print(f"  Patients    : {len(patient_paths)}")
    print(f"  True label  : {true_label}")
    print(f"  Output      : {output_csv}")
    print(f"{'='*60}\n")

    results = []
    correct = 0

    for i, path in enumerate(tqdm(patient_paths, desc=dataset_type)):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        print(f"\n  [{i+1}/{len(patient_paths)}] {name}")

        result               = classify_patient(path, model, processor, proj_out_num)
        result["true_label"] = true_label
        result["correct"]    = result["classification"] == true_label
        if result["correct"]:
            correct += 1

        tick = "✓ CORRECT" if result["correct"] else f"✗ WRONG → {result['classification']}"
        print(f"    Result   : {result['classification'].upper()} [{result['confidence']}]  {tick}")

        for k, v in result.get("steps", {}).items():
            result[k.lower()] = v
        results.append(result)

    fieldnames = ["patient", "true_label", "classification", "confidence",
                  "correct", "modality_used", "step1", "step2", "step3",
                  "step4", "reasoning", "nifti_file"]
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    acc = correct / len(results) * 100 if results else 0
    print(f"\n{'='*60}")
    print(f"  Accuracy    : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Predictions : {dict(Counter(r['classification'] for r in results))}")
    print(f"  Saved to    : {output_csv}")
    print(f"{'='*60}\n")
    return results


# ══════════════════════════════════════════════════════════════════
#  SINGLE PATIENT RUNNER
# ══════════════════════════════════════════════════════════════════

def run_single(path: str, true_label: str = None) -> dict:
    model, processor, proj_out_num = load_model()
    name = Path(path).stem if Path(path).is_file() else Path(path).name

    print(f"\n{'='*60}")
    print(f"  Patient      : {name}")
    print(f"  Model        : Med3DVLM  proj_out_num={proj_out_num}")
    print(f"{'='*60}")

    result = classify_patient(path, model, processor, proj_out_num)

    print(f"\n  Modality       : {result.get('modality_used','N/A')}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")
    if true_label:
        match = "✓ CORRECT" if result["classification"] == true_label else "✗ WRONG"
        print(f"  True label     : {true_label}  →  {match}")

    steps = result.get("steps", {})
    if steps:
        print(f"\n  Step-by-step:")
        for k in sorted(steps.keys()):
            print(f"    {k}: {steps[k]}")

    print(f"\n  Reasoning : {result['reasoning']}")
    print(f"\n  Full response:")
    print(f"  {'-'*55}")
    for line in result["raw_response"].splitlines():
        print(f"  {line}")
    print(f"{'='*60}\n")
    return result


# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE AND RUN
# ══════════════════════════════════════════════════════════════════

RUN_MODE = "single"   # "single" or "batch"

SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000"
SINGLE_LABEL = "meningioma"

BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training"
BATCH_TYPE   = "glioma"
BATCH_LABEL  = "glioma"
BATCH_MAX    = 10
BATCH_OUTPUT = f"{BATCH_TYPE}_med3dvlm_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

if __name__ == "__main__":
    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)
    elif RUN_MODE == "batch":
        run_batch(BATCH_DIR, BATCH_TYPE, BATCH_LABEL,
                  output_csv=BATCH_OUTPUT, max_patients=BATCH_MAX)


  Loading Med3DVLM
  Weights : /home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM
  Device  : cuda:0


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

  proj_out_num : 256
  VRAM     : 30.8GB used | 16.9GB free
  Model ready ✓


  Patient      : BraTS-MEN-00154-000
  Model        : Med3DVLM  proj_out_num=256
    File     : BraTS-MEN-00154-000-t1c.nii.gz  [t1c_brats2024]
    Volume   : (1, 1, 128, 256, 256)  [0.1s]
    Inference: 0.2s

  Modality       : t1c_brats2024
  CLASSIFICATION : UNKNOWN
  CONFIDENCE     : UNKNOWN
  True label     : meningioma  →  ✗ WRONG

  Reasoning : 

  Full response:
  -------------------------------------------------------



In [1]:
"""
Med3DVLM Brain Tumor Classification — Final Correct Pipeline
=============================================================
Requires Python 3.12 (model was built for 3.12, syntax errors on 3.11)

If you are on Python 3.11, create a conda env first:
    conda create -n med3dvlm python=3.12
    conda activate med3dvlm
    pip install -r /home/23bq1a4293/Personal/project/Med3DVLM/requirements.txt
    export PYTHONPATH=/home/23bq1a4293/Personal/project/Med3DVLM:$PYTHONPATH

generate() exact signature (from modeling.py line 1677):
    def generate(self,
        images: Optional[torch.Tensor] = None,   [1,1,128,256,256]
        inputs: Optional[torch.Tensor] = None,   input_ids
        **kwargs                                  passed to super().generate()
    )
    → calls prepare_inputs_for_multimodal internally
    → raises NotImplementedError if inputs_embeds passed directly

Correct call (from app.py):
    model.generate(
        images=image_tensor,
        inputs=input_ids,
        max_new_tokens=512,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
    )

Smart axis detection:
    Uses NIfTI affine Z-row to find true axial axis.
    Then checks which 20% of that axis contains the most non-zero signal
    and centers the crop there — ensures the tumor region is visible.
"""

import os
import re
import csv
import sys
import warnings
import numpy as np
import SimpleITK as sitk

from pathlib import Path
from collections import Counter
from datetime import datetime
from tqdm import tqdm

import torch
from monai.transforms import Resize
from transformers import AutoProcessor, AutoModelForCausalLM

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE PATHS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════

MED3DVLM_REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
MED3DVLM_WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"

DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16
MAX_LENGTH = 512
IMAGE_SIZE = (128, 256, 256)   # exact value from app.py — do not change

# ══════════════════════════════════════════════════════════════════
#  PYTHONPATH SETUP
# ══════════════════════════════════════════════════════════════════

def setup_pythonpath():
    repo = Path(MED3DVLM_REPO)
    if repo.exists():
        for p in [str(repo), str(repo / "src")]:
            if p not in sys.path:
                sys.path.insert(0, p)

setup_pythonpath()

# ══════════════════════════════════════════════════════════════════
#  MODEL LOADER — exact from app.py
# ══════════════════════════════════════════════════════════════════

_model        = None
_processor    = None
_proj_out_num = 256

def load_model():
    global _model, _processor, _proj_out_num

    if _model is not None:
        print("  Model already loaded — reusing")
        return _model, _processor, _proj_out_num

    weights = Path(MED3DVLM_WEIGHTS).resolve()
    if not weights.exists():
        raise FileNotFoundError(
            f"Weights not found at: {weights}\n"
            "Download:\n"
            "  from huggingface_hub import snapshot_download\n"
            "  snapshot_download(\n"
            "      repo_id='MagicXin/Med3DVLM-Qwen-2.5-7B',\n"
            f"     local_dir='{weights}'\n"
            "  )"
        )

    print(f"\n{'='*60}")
    print(f"  Loading Med3DVLM")
    print(f"  Weights : {weights}")
    print(f"  Device  : {DEVICE}")
    print(f"{'='*60}")

    # Exact from app.py
    _processor = AutoProcessor.from_pretrained(
        str(weights),
        max_length=MAX_LENGTH,
        padding_side="right",
        use_fast=False,
        trust_remote_code=True,
        ignore_mismatched_sizes=True,
    )

    _model = AutoModelForCausalLM.from_pretrained(
        str(weights),
        torch_dtype=DTYPE,
        device_map=DEVICE,
        trust_remote_code=True,
    ).to(device=DEVICE).eval()

    # Exact from app.py
    try:
        _proj_out_num = (
            _model.get_model().config.proj_out_num
            if hasattr(_model.get_model().config, "proj_out_num")
            else 256
        )
    except Exception:
        _proj_out_num = 256

    print(f"  proj_out_num : {_proj_out_num}")

    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  VRAM     : {used:.1f}GB used | {total - used:.1f}GB free")

    print("  Model ready ✓\n")
    return _model, _processor, _proj_out_num


# ══════════════════════════════════════════════════════════════════
#  MODALITY FILE FINDER
# ══════════════════════════════════════════════════════════════════

MODALITY_PATTERNS = {
    "t1ce_brats":        re.compile(r".*t1ce\.nii(\.gz)?$",                    re.IGNORECASE),
    "t1c_brats2024":     re.compile(r".*-t1c\.nii(\.gz)?$",                    re.IGNORECASE),
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$",  re.IGNORECASE),
    "healthy":           re.compile(r"^healthy_\d+\.nii(\.gz)?$",              re.IGNORECASE),
}
PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024",
    "pituitary_cecor", "pituitary_cesag", "pituitary_ce3dnav", "healthy",
]
SKIP_KEYWORDS = [
    "seg", "mask", "label", "defacemask", "seed",
    "prediction", "groundtruth", "dwi", "adcb"
]

def find_best_modality_file(path: str) -> tuple:
    p = Path(path)
    if p.is_file():
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(p.name):
                return str(p), key
        if p.name.endswith(".nii") or p.name.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None
    if not p.is_dir():
        return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates
                  if not any(kw in f.name.lower() for kw in SKIP_KEYWORDS)]
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches:
            return str(matches[0]), key
    return (str(candidates[0]), "fallback") if candidates else (None, None)


# ══════════════════════════════════════════════════════════════════
#  SMART IMAGE PREPROCESSING
#
#  The model expects (128, 256, 256) — D × H × W.
#  For BraTS data (240,240,155): axial=axis2, D=155 slices.
#  For pituitary data (512,14,512): very thin on one axis.
#  For healthy data (113,137,113): small isotropic volumes.
#
#  Smart steps:
#   1. Load with SimpleITK (same as app.py)
#   2. Detect true axial axis from NIfTI affine Z-row
#   3. Reorder so axial axis is first (D axis) → (D,H,W)
#   4. Find signal-rich crop: compute mean intensity per axial slice,
#      use sliding window of size 128 to find the 128 consecutive
#      slices with the most signal — centers on tumor region
#   5. Crop to that window, then resize H×W to 256×256 using MONAI
#   6. Return [1, 1, 128, 256, 256] on DEVICE
# ══════════════════════════════════════════════════════════════════

def load_and_preprocess_smart(nifti_path: str) -> tuple[torch.Tensor, dict]:
    """
    Smart preprocessing that finds the signal-rich axial window.
    Returns: (image_tensor [1,1,128,256,256], info_dict)
    """
    # ── Load with SimpleITK (same as app.py) ─────────────────────
    sitk_img  = sitk.ReadImage(str(nifti_path))
    img_array = sitk.GetArrayFromImage(sitk_img).astype(np.float32)
    # SimpleITK gives (Z, Y, X) = (axial_slices, height, width) for standard MRI

    # ── Also load with nibabel for affine-based axis detection ────
    import nibabel as nib
    nib_img    = nib.load(str(nifti_path))
    affine     = nib_img.affine

    # ── Detect true axial axis from affine Z-row ──────────────────
    # Row 2 of affine = world Z (inferior→superior) contribution per voxel axis
    z_row      = np.abs(affine[2, :3])
    axial_axis = int(np.argmax(z_row))   # voxel axis most aligned with world-Z

    # SimpleITK already returns (Z,Y,X) so axis 0 is usually axial for standard MRI
    # But for non-standard orientations we need to reorder
    nib_data   = nib_img.get_fdata().astype(np.float32)
    shape      = nib_data.shape

    # Reorder nibabel data to (D=axial, H, W)
    other_axes = [i for i in range(3) if i != axial_axis]
    nib_reordered = np.transpose(nib_data, [axial_axis] + other_axes)
    D, H, W = nib_reordered.shape

    # ── Find signal-rich crop along D axis ────────────────────────
    # Compute mean intensity per axial slice
    slice_means = nib_reordered.reshape(D, -1).mean(axis=1)

    TARGET_D = IMAGE_SIZE[0]  # 128

    if D <= TARGET_D:
        # Volume has fewer slices than target — use all, pad handled by Resize
        best_start = 0
        best_end   = D
    else:
        # Sliding window of width TARGET_D — find window with max total signal
        window_sums  = np.convolve(slice_means, np.ones(TARGET_D), mode='valid')
        best_start   = int(np.argmax(window_sums))
        best_end     = best_start + TARGET_D

    cropped = nib_reordered[best_start:best_end, :, :]  # (≤128, H, W)

    info = {
        "original_shape": shape,
        "axial_axis":     axial_axis,
        "reordered_shape": nib_reordered.shape,
        "signal_crop":    f"slices {best_start}–{best_end} of {D}",
    }

    # ── Resize to exactly (128, 256, 256) using MONAI ─────────────
    # Add channel dim for MONAI: (1, D, H, W)
    cropped_ch   = np.expand_dims(cropped, axis=0)
    resize_fn    = Resize(spatial_size=IMAGE_SIZE, mode="bilinear")
    resized      = resize_fn(cropped_ch)               # (1, 128, 256, 256)

    # ── To tensor [1, 1, 128, 256, 256] ──────────────────────────
    image_tensor = resized.data.unsqueeze(0).to(device=DEVICE, dtype=DTYPE)

    return image_tensor, info


# ══════════════════════════════════════════════════════════════════
#  CLASSIFICATION PROMPT
# ══════════════════════════════════════════════════════════════════

QUESTION = (
    "You are a consultant neuroradiologist analyzing a full 3D brain MRI "
    "(T1 contrast-enhanced). "
    "Classify into EXACTLY ONE of: glioma | meningioma | pituitary | healthy\n\n"

    "RULE — APPLY FIRST:\n"
    "Does the enhancing mass TOUCH the inner skull surface?\n"
    "  YES → meningioma (extra-axial, dura-based)\n"
    "  NO  → check if inside brain (glioma) or at midline skull base (pituitary)\n\n"

    "GLIOMA: Inside brain parenchyma. Fuzzy infiltrative borders. "
    "Ring enhancement + dark necrotic center (HGG) or absent enhancement (LGG).\n"
    "MENINGIOMA: Outside brain on dura/skull. Sharp clean border. "
    "Solid homogeneous bright enhancement. Dural tail sign.\n"
    "PITUITARY: Midline skull base in sella only. "
    "Snowman shape (macro) or dark spot in bright gland (micro).\n"
    "HEALTHY: No focal mass. Normal symmetric brain. "
    "Normal pituitary enhancement is NOT a tumor.\n\n"

    "Respond in EXACTLY this format, no other text:\n"
    "STEP1: <skull contact test — does mass touch bone?>\n"
    "STEP2: <location — intra-axial / extra-axial / sellar / none>\n"
    "STEP3: <borders — sharp or infiltrative>\n"
    "STEP4: <enhancement — homogeneous solid / ring+necrosis / minimal / absent>\n"
    "CLASSIFICATION: <glioma|meningioma|pituitary|healthy>\n"
    "CONFIDENCE: <high|medium|low>\n"
    "REASONING: <3 sentences citing 3D spatial features, rule out runner-up>"
)


# ══════════════════════════════════════════════════════════════════
#  INFERENCE — exact app.py generate() call
# ══════════════════════════════════════════════════════════════════

def run_inference(image_tensor: torch.Tensor,
                  model, processor, proj_out_num: int) -> str:
    """
    Exact inference pattern from app.py:
        image_tokens = "<im_patch>" * proj_out_num
        input_txt = image_tokens + text_input
        input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"]
        generation = model.generate(
            images=image_input,
            inputs=input_ids,
            max_new_tokens=max_length,
            do_sample=True,
            top_p=0.9,
            temperature=1.0,
        )
        result = tokenizer.decode(generation[0], skip_special_tokens=True)
    """
    # Build prompt — exact app.py pattern
    image_tokens = "<im_patch>" * proj_out_num
    input_txt    = image_tokens + QUESTION
    input_ids    = processor(
        input_txt, return_tensors="pt"
    )["input_ids"].to(device=DEVICE)

    # Generate — exact app.py kwargs
    generation = model.generate(
        images        = image_tensor,
        inputs        = input_ids,
        max_new_tokens= MAX_LENGTH,
        do_sample     = True,
        top_p         = 0.9,
        temperature   = 1.0,
    )

    # Decode — exact app.py
    result = processor.decode(generation[0], skip_special_tokens=True)

    # Extract only the structured response part
    # The model may echo the image tokens and question — find STEP1 or CLASSIFICATION
    for marker in ["STEP1:", "CLASSIFICATION:"]:
        if marker in result:
            result = result[result.find(marker):]
            break

    del image_tensor, input_ids, generation
    torch.cuda.empty_cache()

    return result.strip()


# ══════════════════════════════════════════════════════════════════
#  RESPONSE PARSER
# ══════════════════════════════════════════════════════════════════

def parse_response(raw: str) -> dict:
    classification = "UNKNOWN"
    confidence     = "UNKNOWN"
    reasoning      = ""
    steps          = {}

    lines = raw.splitlines()
    idx   = 0
    while idx < len(lines):
        lc = lines[idx].strip()
        if not lc or ":" not in lc:
            idx += 1
            continue
        key = lc.split(":")[0].strip().upper()
        val = lc.split(":", 1)[1].strip()

        if key == "CLASSIFICATION":
            for cls in ["glioma", "meningioma", "pituitary", "healthy"]:
                if cls in val.lower():
                    classification = cls
                    break
        elif key == "CONFIDENCE":
            for conf in ["high", "medium", "low"]:
                if conf in val.lower():
                    confidence = conf
                    break
        elif key == "REASONING":
            r_lines = [val]
            j = idx + 1
            while j < len(lines):
                nk = lines[j].strip().split(":")[0].strip().upper()
                if nk in ("CLASSIFICATION", "CONFIDENCE",
                          "STEP1", "STEP2", "STEP3", "STEP4"):
                    break
                if lines[j].strip():
                    r_lines.append(lines[j].strip())
                j += 1
            reasoning = " ".join(r_lines)
            idx = j
            continue
        elif key.startswith("STEP"):
            steps[key] = val
        idx += 1

    return {
        "classification": classification,
        "confidence":     confidence,
        "reasoning":      reasoning,
        "steps":          steps,
        "raw_response":   raw,
    }


# ══════════════════════════════════════════════════════════════════
#  PATIENT CLASSIFIER
# ══════════════════════════════════════════════════════════════════

def classify_patient(path: str, model, processor, proj_out_num: int) -> dict:
    import time
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    nifti_path, modality_type = find_best_modality_file(path)
    if nifti_path is None:
        print(f"  WARNING: No NIfTI found in {path}")
        return {
            "patient": patient_name, "nifti_file": None,
            "modality_used": None, "classification": "ERROR",
            "confidence": "N/A", "reasoning": "No NIfTI file found.",
            "steps": {}, "raw_response": "",
        }

    print(f"    File     : {Path(nifti_path).name}  [{modality_type}]")

    # Preprocess with smart axis detection
    t0 = time.time()
    try:
        image_tensor, info = load_and_preprocess_smart(nifti_path)
        print(f"    Shape    : {info['original_shape']} → axis={info['axial_axis']} → {tuple(image_tensor.shape)}")
        print(f"    Crop     : {info['signal_crop']}  [{time.time()-t0:.1f}s]")
    except Exception as e:
        import traceback; traceback.print_exc()
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Preprocessing failed: {e}",
            "steps": {}, "raw_response": "",
        }

    # Inference
    t1 = time.time()
    try:
        raw = run_inference(image_tensor, model, processor, proj_out_num)
        print(f"    Inference: {time.time()-t1:.1f}s")
    except Exception as e:
        import traceback; traceback.print_exc()
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "modality_used": modality_type, "classification": "ERROR",
            "confidence": "N/A", "reasoning": f"Inference failed: {e}",
            "steps": {}, "raw_response": "",
        }

    parsed = parse_response(raw)
    parsed["signal_crop"] = info["signal_crop"]
    parsed["axial_axis"]  = info["axial_axis"]
    return {
        "patient":       patient_name,
        "nifti_file":    nifti_path,
        "modality_used": modality_type,
        **parsed,
    }


# ══════════════════════════════════════════════════════════════════
#  DATASET PATH HELPERS
# ══════════════════════════════════════════════════════════════════

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    base = Path(dataset_dir)
    if dataset_type == "glioma":
        paths = []
        for grade in ["HGG", "LGG"]:
            gd = base / grade
            if gd.exists():
                paths += [str(d) for d in sorted(gd.iterdir()) if d.is_dir()]
        return paths or [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "pituitary":
        paths = []
        for sub in sorted(base.iterdir()):
            if sub.is_dir() and sub.name.startswith("sub-"):
                anat = sub / "anat"
                paths.append(str(anat) if anat.exists() else str(sub))
        return paths
    elif dataset_type == "healthy":
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])
    else:
        raise ValueError(f"Unknown dataset_type: '{dataset_type}'")


# ══════════════════════════════════════════════════════════════════
#  BATCH RUNNER
# ══════════════════════════════════════════════════════════════════

def run_batch(dataset_dir: str, dataset_type: str, true_label: str,
              output_csv: str = "results.csv",
              max_patients: int = None) -> list:

    model, processor, proj_out_num = load_model()
    patient_paths = get_patient_paths(dataset_dir, dataset_type)
    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*60}")
    print(f"  BATCH       : {dataset_type.upper()}")
    print(f"  Patients    : {len(patient_paths)}")
    print(f"  True label  : {true_label}")
    print(f"  Output      : {output_csv}")
    print(f"{'='*60}\n")

    results = []
    correct = 0

    for i, path in enumerate(tqdm(patient_paths, desc=dataset_type)):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        print(f"\n  [{i+1}/{len(patient_paths)}] {name}")

        result               = classify_patient(path, model, processor, proj_out_num)
        result["true_label"] = true_label
        result["correct"]    = result["classification"] == true_label
        if result["correct"]:
            correct += 1

        tick = "✓ CORRECT" if result["correct"] else f"✗ WRONG → {result['classification']}"
        print(f"    Result   : {result['classification'].upper()} [{result['confidence']}]  {tick}")

        for k, v in result.get("steps", {}).items():
            result[k.lower()] = v
        results.append(result)

    fieldnames = ["patient", "true_label", "classification", "confidence",
                  "correct", "modality_used", "axial_axis", "signal_crop",
                  "step1", "step2", "step3", "step4", "reasoning", "nifti_file"]
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    acc = correct / len(results) * 100 if results else 0
    print(f"\n{'='*60}")
    print(f"  Accuracy    : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Predictions : {dict(Counter(r['classification'] for r in results))}")
    print(f"  Saved to    : {output_csv}")
    print(f"{'='*60}\n")
    return results


# ══════════════════════════════════════════════════════════════════
#  SINGLE PATIENT RUNNER
# ══════════════════════════════════════════════════════════════════

def run_single(path: str, true_label: str = None) -> dict:
    model, processor, proj_out_num = load_model()
    name = Path(path).stem if Path(path).is_file() else Path(path).name

    print(f"\n{'='*60}")
    print(f"  Patient      : {name}")
    print(f"  Model        : Med3DVLM  proj_out_num={proj_out_num}")
    print(f"{'='*60}")

    result = classify_patient(path, model, processor, proj_out_num)

    print(f"\n  Modality       : {result.get('modality_used','N/A')}")
    print(f"  Axial axis     : {result.get('axial_axis','N/A')}")
    print(f"  Signal crop    : {result.get('signal_crop','N/A')}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")
    if true_label:
        match = "✓ CORRECT" if result["classification"] == true_label else "✗ WRONG"
        print(f"  True label     : {true_label}  →  {match}")

    steps = result.get("steps", {})
    if steps:
        print(f"\n  Step-by-step:")
        for k in sorted(steps.keys()):
            print(f"    {k}: {steps[k]}")

    print(f"\n  Reasoning : {result['reasoning']}")
    print(f"\n  Full response:")
    print(f"  {'-'*55}")
    for line in result["raw_response"].splitlines():
        print(f"  {line}")
    print(f"{'='*60}\n")
    return result


# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE AND RUN
# ══════════════════════════════════════════════════════════════════

RUN_MODE = "single"   # "single" or "batch"

SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000"
SINGLE_LABEL = "meningioma"

BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training"
BATCH_TYPE   = "glioma"
BATCH_LABEL  = "glioma"
BATCH_MAX    = 10
BATCH_OUTPUT = f"{BATCH_TYPE}_med3dvlm_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

if __name__ == "__main__":
    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)
    elif RUN_MODE == "batch":
        run_batch(BATCH_DIR, BATCH_TYPE, BATCH_LABEL,
                  output_csv=BATCH_OUTPUT, max_patients=BATCH_MAX)

/home/23bq1a4293/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



  Loading Med3DVLM
  Weights : /home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM
  Device  : cuda:0


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

  proj_out_num : 256
  VRAM     : 15.4GB used | 32.3GB free
  Model ready ✓


  Patient      : BraTS-MEN-00154-000
  Model        : Med3DVLM  proj_out_num=256
    File     : BraTS-MEN-00154-000-t1c.nii.gz  [t1c_brats2024]
    Shape    : (240, 240, 155) → axis=2 → (1, 1, 128, 256, 256)
    Crop     : slices 9–137 of 155  [1.0s]


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


    Inference: 4.3s

  Modality       : t1c_brats2024
  Axial axis     : 2
  Signal crop    : slices 9–137 of 155
  CLASSIFICATION : UNKNOWN
  CONFIDENCE     : UNKNOWN
  True label     : meningioma  →  ✗ WRONG

  Reasoning : 

  Full response:
  -------------------------------------------------------



In [3]:
import os, sys, torch, numpy as np, SimpleITK as sitk, nibabel as nib
from monai.transforms import Resize
from transformers import AutoProcessor, AutoModelForCausalLM

WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"
REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
DEVICE  = "cuda:0"
DTYPE   = torch.bfloat16

for p in [REPO, REPO+"/src"]:
    if p not in sys.path: sys.path.insert(0, p)

processor = AutoProcessor.from_pretrained(
    WEIGHTS, max_length=512, padding_side="right",
    use_fast=False, trust_remote_code=True, ignore_mismatched_sizes=True,
)
model = AutoModelForCausalLM.from_pretrained(
    WEIGHTS, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True,
).to(DEVICE).eval()

try:
    proj_out_num = model.get_model().config.proj_out_num if hasattr(model.get_model().config, "proj_out_num") else 256
except:
    proj_out_num = 256
print(f"proj_out_num={proj_out_num}")

# Load volume
nifti_path = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000/BraTS-MEN-00154-000-t1c.nii.gz"
nib_img  = nib.load(nifti_path)
data     = nib_img.get_fdata().astype(np.float32)
z_row    = np.abs(nib_img.affine[2,:3])
ax       = int(np.argmax(z_row))
other    = [i for i in range(3) if i != ax]
data     = np.transpose(data, [ax]+other)   # (D,H,W)
D        = data.shape[0]
smeans   = data.reshape(D,-1).mean(1)
win      = np.convolve(smeans, np.ones(128), mode='valid')
s        = int(np.argmax(win))
cropped  = data[s:s+128]
ch       = np.expand_dims(cropped, 0)
resized  = Resize(spatial_size=(128,256,256), mode="bilinear")(ch)
img_t    = resized.data.unsqueeze(0).to(DEVICE, dtype=DTYPE)
print(f"image tensor: {img_t.shape}")

# Simple question first
image_tokens = "<im_patch>" * proj_out_num
question     = "Describe what you see in this brain MRI scan."
input_txt    = image_tokens + question
input_ids    = processor(input_txt, return_tensors="pt")["input_ids"].to(DEVICE)
print(f"input_ids shape: {input_ids.shape}")

with torch.no_grad():
    out = model.generate(
        images=img_t, inputs=input_ids,
        max_new_tokens=200, do_sample=True, top_p=0.9, temperature=1.0,
    )

print(f"output shape: {out.shape}")
print(f"output tokens: {out[0].tolist()}")
decoded = processor.decode(out[0], skip_special_tokens=True)
print(f"\nDECODED:\n{decoded}")
decoded_full = processor.decode(out[0], skip_special_tokens=False)
print(f"\nWITH SPECIAL TOKENS (first 300 chars):\n{decoded_full[:300]}")

Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

proj_out_num=256
image tensor: torch.Size([1, 1, 128, 256, 256])
input_ids shape: torch.Size([1, 266])
output shape: torch.Size([1, 4])
output tokens: [2160, 2407, 685, 151645]

DECODED:
 Ischemia

WITH SPECIAL TOKENS (first 300 chars):
 Ischemia<|im_end|>


In [4]:
import sys
import torch
import torch.nn.functional as F
import nibabel as nib
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Your Local Configuration ---
WEIGHTS = "/home/23bq1a4293/Med3DVLM/downloaded_models/Med3DVLM"
REPO    = "/home/23bq1a4293/Personal/project/Med3DVLM"
DEVICE  = "cuda:0"
DTYPE   = torch.bfloat16
TARGET_SCAN = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000/BraTS-MEN-00154-000-t1c.nii.gz"

# Ensure Python can find the local Med3DVLM repository modules
sys.path.append(REPO)
# --------------------------------

def load_and_preprocess_nifti(file_path):
    """Loads, normalizes, and resizes a 3D NIfTI volume."""
    # 1. Load data
    nii_img = nib.load(file_path)
    volume_data = nii_img.get_fdata()
    
    # 2. Normalize (Min-Max scaling to 0-1)
    min_val = np.min(volume_data)
    max_val = np.max(volume_data)
    normalized_volume = (volume_data - min_val) / (max_val - min_val + 1e-8)
    
    # 3. Convert to Tensor & add Batch/Channel dimensions -> (1, 1, D, H, W)
    tensor_volume = torch.tensor(normalized_volume, dtype=torch.float32)
    tensor_volume = tensor_volume.unsqueeze(0).unsqueeze(0) 
    
    # 4. Resize the volume to match Med3DVLM's expected input shape
    # NOTE: If you still get a shape error, try changing this to (32, 224, 224) 
    # based on the model's specific training configuration.
    TARGET_SHAPE = (128, 256, 256) 
    
    tensor_volume = F.interpolate(
        tensor_volume, 
        size=TARGET_SHAPE, 
        mode='trilinear', 
        align_corners=False
    )
    
    return tensor_volume

def generate_summary(nifti_path):
    print(f"Using device: {DEVICE} with precision: {DTYPE}")

    # ==========================================
    # 1. Load Model & Tokenizer
    # ==========================================
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(WEIGHTS, trust_remote_code=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        WEIGHTS, 
        torch_dtype=DTYPE, 
        trust_remote_code=True
    ).to(DEVICE)
    model.eval()

    # ==========================================
    # 2. Prepare the 3D Image & Prompt
    # ==========================================
    print(f"Processing NIfTI file: {nifti_path}")
    image_tensor = load_and_preprocess_nifti(nifti_path).to(DEVICE, dtype=DTYPE)
    
    prompt = "<image>\nPlease provide a short summary of the findings in this 3D medical scan."
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # ==========================================
    # 3. Generate Output
    # ==========================================
    print("Generating summary...")
    with torch.no_grad():
        output_ids = model.generate(
            inputs=inputs.input_ids,               # Fixed argument name
            attention_mask=inputs.attention_mask,  # Fixed argument name
            images=image_tensor,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.3,
            top_p=0.9
        )
    
    # Decode the generated tokens
    input_len = inputs.input_ids.shape[1]
    generated_text = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    
    print("\n--- Model Summary ---")
    print(generated_text.strip())
    print("---------------------")

if __name__ == "__main__":
    generate_summary(TARGET_SCAN)

Using device: cuda:0 with precision: torch.bfloat16
Loading model and tokenizer...


Loading weights:   0%|          | 0/813 [00:00<?, ?it/s]

Processing NIfTI file: /home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00154-000/BraTS-MEN-00154-000-t1c.nii.gz


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating summary...

--- Model Summary ---

---------------------


In [4]:
"""
Brain Tumor Radiomic Classifier — Inference Script
====================================================
Use this after training with brain_tumor_radiomic_classifier.py

Modes:
  1. SINGLE   — classify one patient folder or .nii.gz file
  2. BATCH    — classify all patients in a dataset folder
  3. EVALUATE — batch with known true labels + accuracy report

Requirements:
  pip install nibabel numpy scipy scikit-learn pandas joblib tqdm

Usage:
  Set RUN_MODE and paths at the bottom, then run.
"""

import re
import csv
import warnings
import numpy as np
import nibabel as nib

from pathlib import Path
from collections import Counter
from datetime import datetime
from tqdm import tqdm
from scipy import ndimage
from scipy.stats import skew, kurtosis

import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
#  ★ EDIT THESE
# ══════════════════════════════════════════════════════════════════

MODEL_PKL = "./radiomic_output/brain_tumor_classifier.pkl"  # trained model

# Single patient
SINGLE_PATH  = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train/BraTS-MEN-00152-000"
SINGLE_LABEL = "meningioma"   # set None if unknown

# Batch / Evaluate
BATCH_DIR    = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training"
BATCH_TYPE   = "glioma"       # "glioma" | "meningioma" | "pituitary" | "healthy"
BATCH_LABEL  = "glioma"       # true label for accuracy — set None if unknown
BATCH_MAX    = None           # None = all patients
BATCH_OUTPUT = f"{BATCH_TYPE}_inference_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

RUN_MODE = "batch"           # "single" | "batch" | "evaluate_all"

# ══════════════════════════════════════════════════════════════════
#  MODALITY FILE FINDER
# ══════════════════════════════════════════════════════════════════

MODALITY_PATTERNS = {
    "t1ce_brats":        re.compile(r".*t1ce\.nii(\.gz)?$",                    re.IGNORECASE),
    "t1c_brats2024":     re.compile(r".*-t1c\.nii(\.gz)?$",                    re.IGNORECASE),
    "pituitary_cecor":   re.compile(r".*acq-CECor_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_cesag":   re.compile(r".*acq-CESag_T1w\.nii(\.gz)?$",           re.IGNORECASE),
    "pituitary_ce3dnav": re.compile(r".*acq-CE3DNavigation_T1w\.nii(\.gz)?$",  re.IGNORECASE),
    "healthy":           re.compile(r"^healthy_\d+\.nii(\.gz)?$",              re.IGNORECASE),
}
PATTERN_PRIORITY = [
    "t1ce_brats", "t1c_brats2024",
    "pituitary_cecor", "pituitary_cesag", "pituitary_ce3dnav", "healthy",
]
SKIP_KEYWORDS = ["seg","mask","label","defacemask","seed","prediction","groundtruth","dwi","adcb"]

def find_best_modality_file(path: str) -> tuple:
    p = Path(path)
    if p.is_file():
        for key in PATTERN_PRIORITY:
            if MODALITY_PATTERNS[key].match(p.name):
                return str(p), key
        if p.name.endswith(".nii") or p.name.endswith(".nii.gz"):
            return str(p), "direct_file"
        return None, None
    if not p.is_dir():
        return None, None
    candidates = sorted(list(p.rglob("*.nii")) + list(p.rglob("*.nii.gz")))
    candidates = [f for f in candidates if not any(kw in f.name.lower() for kw in SKIP_KEYWORDS)]
    for key in PATTERN_PRIORITY:
        matches = [f for f in candidates if MODALITY_PATTERNS[key].match(f.name)]
        if matches:
            return str(matches[0]), key
    return (str(candidates[0]), "fallback") if candidates else (None, None)


# ══════════════════════════════════════════════════════════════════
#  DATASET PATH HELPERS
# ══════════════════════════════════════════════════════════════════

def get_patient_paths(dataset_dir: str, dataset_type: str) -> list:
    base = Path(dataset_dir)
    if dataset_type == "glioma":
        paths = []
        for grade in ["HGG", "LGG"]:
            gd = base / grade
            if gd.exists():
                paths += [str(d) for d in sorted(gd.iterdir()) if d.is_dir()]
        return paths or [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "meningioma":
        return [str(d) for d in sorted(base.iterdir()) if d.is_dir()]
    elif dataset_type == "pituitary":
        paths = []
        for sub in sorted(base.iterdir()):
            if sub.is_dir() and sub.name.startswith("sub-"):
                anat = sub / "anat"
                paths.append(str(anat) if anat.exists() else str(sub))
        return paths
    elif dataset_type == "healthy":
        return sorted([str(f) for f in base.glob("healthy_*.nii.gz")])
    else:
        raise ValueError(f"Unknown dataset_type: '{dataset_type}'")


# ══════════════════════════════════════════════════════════════════
#  FEATURE EXTRACTION  (must match training exactly)
# ══════════════════════════════════════════════════════════════════

def extract_features(nifti_path: str) -> dict:
    """Extract the same 53 features used during training."""
    features = {}
    try:
        img  = nib.load(nifti_path)
        data = img.get_fdata().astype(np.float32)
        if data.ndim == 4:
            data = data[..., 0]

        affine = img.affine
        vox    = np.abs(np.diag(affine)[:3])
        shape  = data.shape

        z_row      = np.abs(affine[2, :3])
        axial_axis = int(np.argmax(z_row))
        features["axial_axis"] = float(axial_axis)

        nonzero = data[data > 0]
        if len(nonzero) < 100:
            return None

        p1  = float(np.percentile(nonzero, 1))
        p99 = float(np.percentile(nonzero, 99))
        norm = np.clip(data, p1, p99)
        norm = (norm - p1) / (p99 - p1 + 1e-8)
        nz   = norm[norm > 0]

        features["mean"]     = float(nz.mean())
        features["std"]      = float(nz.std())
        features["median"]   = float(np.median(nz))
        features["p10"]      = float(np.percentile(nz, 10))
        features["p25"]      = float(np.percentile(nz, 25))
        features["p75"]      = float(np.percentile(nz, 75))
        features["p90"]      = float(np.percentile(nz, 90))
        features["p95"]      = float(np.percentile(nz, 95))
        features["p99_val"]  = float(np.percentile(nz, 99))
        features["iqr"]      = float(np.percentile(nz,75) - np.percentile(nz,25))
        features["skewness"] = float(skew(nz))
        features["kurtosis"] = float(kurtosis(nz))
        features["cv"]       = float(nz.std() / (nz.mean() + 1e-8))
        features["nz_frac"]  = float(len(nz) / data.size)

        hi_thresh = float(np.percentile(nz, 95))
        hi_mask   = norm > hi_thresh
        features["hi_count"] = float(hi_mask.sum())
        features["hi_frac"]  = float(hi_mask.sum() / max(len(nz), 1))

        labeled, n_comp = ndimage.label(hi_mask)
        features["n_components"] = float(n_comp)

        if n_comp > 0:
            sizes       = [int((labeled == i).sum()) for i in range(1, n_comp+1)]
            largest_idx = int(np.argmax(sizes)) + 1
            main_mask   = labeled == largest_idx
            main_size   = sizes[largest_idx - 1]
            vox_vol     = float(vox[0] * vox[1] * vox[2])
            features["lesion_vol_mm3"] = float(main_size * vox_vol)

            coords   = np.argwhere(main_mask)
            centroid = coords.mean(axis=0)
            norm_c   = centroid / np.array(shape)
            features["centroid_lr"]  = float(norm_c[0])
            features["centroid_ap"]  = float(norm_c[1])
            features["centroid_si"]  = float(norm_c[axial_axis])

            extent    = coords.max(axis=0) - coords.min(axis=0)
            extent_mm = extent * vox
            features["extent_d0_mm"]  = float(extent_mm[0])
            features["extent_d1_mm"]  = float(extent_mm[1])
            features["extent_d2_mm"]  = float(extent_mm[2])
            features["max_extent_mm"] = float(extent_mm.max())

            dist_to_edge = float(min(
                norm_c[0], 1-norm_c[0],
                norm_c[1], 1-norm_c[1],
                norm_c[2], 1-norm_c[2]
            ))
            features["dist_to_edge"] = dist_to_edge

            main_vals = norm[main_mask]
            features["lesion_mean"]     = float(main_vals.mean())
            features["lesion_std"]      = float(main_vals.std())
            features["lesion_cv"]       = float(main_vals.std() / (main_vals.mean() + 1e-8))
            features["lesion_skewness"] = float(skew(main_vals))
            features["lesion_p95"]      = float(np.percentile(main_vals, 95))

            try:
                core_mask = ndimage.binary_erosion(main_mask, iterations=4)
                rim_mask  = main_mask & ~core_mask
                core_mean = float(norm[core_mask].mean()) if core_mask.any() else features["lesion_mean"]
                rim_mean  = float(norm[rim_mask].mean())  if rim_mask.any()  else features["lesion_mean"]
                features["rim_core_ratio"] = float(rim_mean / (core_mean + 1e-8))
                features["core_mean"]      = core_mean
                features["rim_mean"]       = rim_mean
            except Exception:
                features["rim_core_ratio"] = 1.0
                features["core_mean"]      = features["lesion_mean"]
                features["rim_mean"]       = features["lesion_mean"]

            bbox_vol = float(extent_mm.prod()) if extent_mm.prod() > 0 else 1.0
            features["sphericity"] = float(features["lesion_vol_mm3"] / (bbox_vol + 1e-8))
        else:
            for k in ["lesion_vol_mm3","centroid_lr","centroid_ap","centroid_si",
                      "extent_d0_mm","extent_d1_mm","extent_d2_mm","max_extent_mm",
                      "dist_to_edge","lesion_mean","lesion_std","lesion_cv",
                      "lesion_skewness","lesion_p95","rim_core_ratio",
                      "core_mean","rim_mean","sphericity"]:
                features[k] = 0.0

        mid    = shape[0] // 2
        l_mean = float(norm[:mid].mean())
        r_mean = float(norm[mid:].mean())
        features["asym_lr"] = float(abs(l_mean - r_mean) / (max(l_mean, r_mean) + 1e-8))
        features["l_mean"]  = l_mean
        features["r_mean"]  = r_mean

        sx = slice(int(shape[0]*0.40), int(shape[0]*0.60))
        sy = slice(int(shape[1]*0.35), int(shape[1]*0.65))
        sz = slice(int(shape[2]*0.10), int(shape[2]*0.35))
        sellar = norm[sx, sy, sz]
        features["sellar_mean"]  = float(sellar.mean())
        features["sellar_max"]   = float(sellar.max())
        features["sellar_ratio"] = float(sellar.max() / (hi_thresh + 1e-8))

        sup_ax = shape[axial_axis]
        if axial_axis == 2:
            sup = norm[:, :, int(sup_ax*0.55):]
        elif axial_axis == 1:
            sup = norm[:, int(sup_ax*0.55):, :]
        else:
            sup = norm[int(sup_ax*0.55):, :, :]
        features["superior_mean"] = float(sup[sup > hi_thresh].mean()) if (sup > hi_thresh).any() else 0.0
        features["superior_frac"] = float((sup > hi_thresh).sum() / max((norm > hi_thresh).sum(), 1))

        if n_comp > 0:
            gx = ndimage.sobel(norm, axis=0)
            gy = ndimage.sobel(norm, axis=1)
            gz = ndimage.sobel(norm, axis=2)
            grad_mag = np.sqrt(gx**2 + gy**2 + gz**2)
            dilated   = ndimage.binary_dilation(main_mask, iterations=2)
            boundary  = dilated & ~main_mask
            features["boundary_gradient"] = float(grad_mag[boundary].mean()) if boundary.any() else 0.0
            features["interior_gradient"] = float(grad_mag[main_mask].mean()) if main_mask.any() else 0.0
            features["gradient_ratio"]    = float(
                features["boundary_gradient"] / (features["interior_gradient"] + 1e-8)
            )
        else:
            features["boundary_gradient"] = 0.0
            features["interior_gradient"] = 0.0
            features["gradient_ratio"]    = 0.0

        features["vox_x"]    = float(vox[0])
        features["vox_y"]    = float(vox[1])
        features["vox_z"]    = float(vox[2])
        features["shape_d0"] = float(shape[0])
        features["shape_d1"] = float(shape[1])
        features["shape_d2"] = float(shape[2])

    except Exception as e:
        print(f"      Feature extraction error: {e}")
        return None

    return features


# ══════════════════════════════════════════════════════════════════
#  MODEL LOADER
# ══════════════════════════════════════════════════════════════════

_trained_model = None
_label_encoder = None
_feature_cols  = None

def load_trained_model(model_pkl: str):
    global _trained_model, _label_encoder, _feature_cols

    if _trained_model is not None:
        return _trained_model, _label_encoder, _feature_cols

    if not Path(model_pkl).exists():
        raise FileNotFoundError(
            f"Model not found: {model_pkl}\n"
            "Train first by running brain_tumor_radiomic_classifier.py"
        )

    _trained_model, _label_encoder = joblib.load(model_pkl)
    print(f"  Model loaded : {model_pkl}")
    print(f"  Classes      : {list(_label_encoder.classes_)}")

    # Feature column order must match training — derive from a dummy extraction
    dummy_features = extract_features.__code__.co_consts
    # Use a fixed known order matching the training script
    _feature_cols = [
        "axial_axis","mean","std","median","p10","p25","p75","p90","p95","p99_val",
        "iqr","skewness","kurtosis","cv","nz_frac","hi_count","hi_frac","n_components",
        "lesion_vol_mm3","centroid_lr","centroid_ap","centroid_si",
        "extent_d0_mm","extent_d1_mm","extent_d2_mm","max_extent_mm","dist_to_edge",
        "lesion_mean","lesion_std","lesion_cv","lesion_skewness","lesion_p95",
        "rim_core_ratio","core_mean","rim_mean","sphericity",
        "asym_lr","l_mean","r_mean",
        "sellar_mean","sellar_max","sellar_ratio",
        "superior_mean","superior_frac",
        "boundary_gradient","interior_gradient","gradient_ratio",
        "vox_x","vox_y","vox_z","shape_d0","shape_d1","shape_d2",
    ]
    print(f"  Feature cols : {len(_feature_cols)}")
    return _trained_model, _label_encoder, _feature_cols


# ══════════════════════════════════════════════════════════════════
#  CORE PREDICT FUNCTION
# ══════════════════════════════════════════════════════════════════

def predict_one(path: str, model, label_encoder, feature_cols: list) -> dict:
    """Predict class for one patient path (folder or .nii.gz file)."""
    import time
    patient_name = Path(path).stem if Path(path).is_file() else Path(path).name

    # Find NIfTI
    nifti_path, modality = find_best_modality_file(path)
    if nifti_path is None:
        return {
            "patient": patient_name, "nifti_file": None,
            "classification": "ERROR", "confidence": "N/A",
            "error": "No NIfTI file found",
        }

    # Extract features
    t0    = time.time()
    feats = extract_features(nifti_path)
    t_feat = time.time() - t0

    if feats is None:
        return {
            "patient": patient_name, "nifti_file": nifti_path,
            "classification": "ERROR", "confidence": "N/A",
            "error": "Feature extraction failed",
        }

    # Build feature vector in correct column order
    X_new = np.array([[feats.get(col, 0.0) for col in feature_cols]], dtype=np.float32)

    # Predict
    pred_idx   = model.predict(X_new)[0]
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    # Probabilities (SVM with probability=True)
    proba_dict = {}
    if hasattr(model, "predict_proba"):
        try:
            proba    = model.predict_proba(X_new)[0]
            max_prob = float(proba.max())
            confidence = "high" if max_prob > 0.70 else "medium" if max_prob > 0.50 else "low"
            for i, cls in enumerate(label_encoder.classes_):
                proba_dict[f"prob_{cls}"] = float(proba[i])
        except Exception:
            max_prob   = 1.0
            confidence = "unknown"
    else:
        confidence = "unknown"
        max_prob   = 1.0

    result = {
        "patient":        patient_name,
        "nifti_file":     nifti_path,
        "modality":       modality,
        "classification": pred_label,
        "confidence":     confidence,
        "feat_time_s":    round(t_feat, 2),
        **proba_dict,
        # Key features that drove the decision (for interpretability)
        "dist_to_edge":   round(feats.get("dist_to_edge", 0), 3),
        "rim_core_ratio": round(feats.get("rim_core_ratio", 0), 3),
        "lesion_cv":      round(feats.get("lesion_cv", 0), 3),
        "sellar_ratio":   round(feats.get("sellar_ratio", 0), 3),
        "centroid_si":    round(feats.get("centroid_si", 0), 3),
        "asym_lr":        round(feats.get("asym_lr", 0), 3),
    }
    return result


# ══════════════════════════════════════════════════════════════════
#  SINGLE PATIENT MODE
# ══════════════════════════════════════════════════════════════════

def run_single(path: str, true_label: str = None):
    model, le, feat_cols = load_trained_model(MODEL_PKL)

    name = Path(path).stem if Path(path).is_file() else Path(path).name
    print(f"\n{'='*55}")
    print(f"  Patient : {name}")
    print(f"{'='*55}")

    result = predict_one(path, model, le, feat_cols)

    print(f"\n  File           : {Path(result['nifti_file']).name if result.get('nifti_file') else 'NOT FOUND'}")
    print(f"  Modality       : {result.get('modality','N/A')}")
    print(f"  CLASSIFICATION : {result['classification'].upper()}")
    print(f"  CONFIDENCE     : {result['confidence']}")

    if true_label:
        match = "✓ CORRECT" if result["classification"] == true_label else "✗ WRONG"
        print(f"  True label     : {true_label}  →  {match}")

    # Print class probabilities
    prob_keys = [k for k in result if k.startswith("prob_")]
    if prob_keys:
        print(f"\n  Class probabilities:")
        for k in sorted(prob_keys, key=lambda x: -result[x]):
            bar_len = int(result[k] * 30)
            bar = "█" * bar_len + "░" * (30 - bar_len)
            print(f"    {k[5:]:<12} {bar} {result[k]*100:5.1f}%")

    # Print key radiomic features that determined classification
    print(f"\n  Key features:")
    print(f"    dist_to_edge   : {result['dist_to_edge']:.3f}  "
          f"({'near surface → meningioma' if result['dist_to_edge'] < 0.18 else 'deep → glioma/pituitary'})")
    print(f"    rim_core_ratio : {result['rim_core_ratio']:.3f}  "
          f"({'ring pattern → glioma HGG' if result['rim_core_ratio'] > 1.6 else 'solid mass'})")
    print(f"    lesion_cv      : {result['lesion_cv']:.3f}  "
          f"({'heterogeneous' if result['lesion_cv'] > 0.25 else 'homogeneous'})")
    print(f"    sellar_ratio   : {result['sellar_ratio']:.3f}  "
          f"({'pituitary signal' if result['sellar_ratio'] > 0.85 else 'no sellar signal'})")
    print(f"    centroid_si    : {result['centroid_si']:.3f}  "
          f"({'inferior skull base → pituitary' if result['centroid_si'] < 0.30 else 'upper brain'})")
    print(f"    asym_lr        : {result['asym_lr']:.3f}  "
          f"({'asymmetric' if result['asym_lr'] > 0.05 else 'symmetric'})")
    print(f"\n  Feature extraction time: {result['feat_time_s']}s")
    print(f"{'='*55}\n")

    return result


# ══════════════════════════════════════════════════════════════════
#  BATCH / EVALUATE MODE
# ══════════════════════════════════════════════════════════════════

def run_batch(dataset_dir: str, dataset_type: str,
              true_label: str = None,
              output_csv: str = "inference_results.csv",
              max_patients: int = None) -> list:

    model, le, feat_cols = load_trained_model(MODEL_PKL)
    patient_paths = get_patient_paths(dataset_dir, dataset_type)
    if max_patients:
        patient_paths = patient_paths[:max_patients]

    print(f"\n{'='*55}")
    print(f"  BATCH INFERENCE : {dataset_type.upper()}")
    print(f"  Patients        : {len(patient_paths)}")
    print(f"  True label      : {true_label or 'unknown'}")
    print(f"  Output          : {output_csv}")
    print(f"{'='*55}\n")

    results  = []
    correct  = 0
    pred_all = []

    for i, path in enumerate(tqdm(patient_paths, desc=dataset_type)):
        name = Path(path).stem if Path(path).is_file() else Path(path).name
        result = predict_one(path, model, le, feat_cols)

        result["true_label"] = true_label or "unknown"
        result["correct"]    = (result["classification"] == true_label) if true_label else None
        if result["correct"]:
            correct += 1

        pred_all.append(result["classification"])

        tick = ""
        if true_label:
            tick = "✓" if result["correct"] else f"✗ → {result['classification']}"
        print(f"  [{i+1:3d}/{len(patient_paths)}] {name:<40} "
              f"{result['classification'].upper():<12} [{result['confidence']}]  {tick}")

        results.append(result)

    # Save CSV
    all_keys = set()
    for r in results:
        all_keys.update(r.keys())
    fieldnames = sorted(all_keys)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)

    # Summary
    pred_counts = dict(Counter(pred_all))
    print(f"\n{'='*55}")
    print(f"  INFERENCE SUMMARY")
    print(f"{'='*55}")
    print(f"  Total patients   : {len(results)}")
    print(f"  Predictions      : {pred_counts}")
    if true_label:
        acc = correct / len(results) * 100 if results else 0
        print(f"  Correct          : {correct}/{len(results)} = {acc:.1f}%")
    print(f"  Saved to         : {output_csv}")
    print(f"{'='*55}\n")

    return results


# ══════════════════════════════════════════════════════════════════
#  EVALUATE ALL 4 CLASSES
# ══════════════════════════════════════════════════════════════════

DATA_ROOTS = {
    "glioma":     "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training",
    "meningioma": "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train",
    "pituitary":  "/home/23bq1a4293/Personal/project/seg_data/pitutary/pituitary_tumor_dataset",
    "healthy":    "/home/23bq1a4293/Personal/project/seg_data/healthy",
}

def run_evaluate_all(max_per_class: int = 20):
    """Evaluate all 4 classes and print a combined report."""
    from sklearn.metrics import classification_report, confusion_matrix

    model, le, feat_cols = load_trained_model(MODEL_PKL)

    print(f"\n{'='*55}")
    print(f"  FULL EVALUATION — ALL 4 CLASSES")
    print(f"  Patients per class: {max_per_class}")
    print(f"{'='*55}\n")

    all_true = []
    all_pred = []
    all_results = []

    for label, data_root in DATA_ROOTS.items():
        if not Path(data_root).exists():
            print(f"  SKIP: {label} — path not found")
            continue

        paths = get_patient_paths(data_root, label)[:max_per_class]
        print(f"\n  [{label.upper()}] — {len(paths)} patients")

        for path in tqdm(paths, desc=f"  {label}", leave=False):
            result = predict_one(path, model, le, feat_cols)
            result["true_label"] = label
            result["correct"]    = result["classification"] == label
            all_true.append(label)
            all_pred.append(result["classification"])
            all_results.append(result)

            tick = "✓" if result["correct"] else f"✗ → {result['classification']}"
            name = Path(path).stem if Path(path).is_file() else Path(path).name
            print(f"    {name:<40} {result['classification'].upper():<12} {tick}")

    # Report
    print(f"\n{'='*55}")
    print(f"  CLASSIFICATION REPORT")
    print(f"{'='*55}")
    classes = sorted(set(all_true))
    print(classification_report(all_true, all_pred, target_names=classes, digits=3))

    # Confusion matrix
    cm = confusion_matrix(all_true, all_pred, labels=classes)
    print(f"  Confusion matrix (rows=true, cols=pred):")
    header = f"  {'':>12}" + "".join(f"{c[:6]:>8}" for c in classes)
    print(header)
    for i, row_label in enumerate(classes):
        row = f"  {row_label:>12}" + "".join(f"{v:>8}" for v in cm[i])
        print(row)

    acc = sum(t == p for t, p in zip(all_true, all_pred)) / len(all_true) * 100
    print(f"\n  Overall accuracy : {acc:.1f}%")

    # Save
    out_csv = f"full_evaluation_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    all_keys = set()
    for r in all_results:
        all_keys.update(r.keys())
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=sorted(all_keys), extrasaction="ignore")
        writer.writeheader()
        writer.writerows(all_results)
    print(f"  Results saved    : {out_csv}")
    print(f"{'='*55}\n")

    return all_results


# ══════════════════════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    if RUN_MODE == "single":
        run_single(SINGLE_PATH, SINGLE_LABEL)

    elif RUN_MODE == "batch":
        run_batch(
            dataset_dir  = BATCH_DIR,
            dataset_type = BATCH_TYPE,
            true_label   = BATCH_LABEL,
            output_csv   = BATCH_OUTPUT,
            max_patients = BATCH_MAX,
        )

    elif RUN_MODE == "evaluate_all":
        run_evaluate_all(max_per_class=20)   # 20 per class = 80 patients total

    else:
        print(f"Unknown RUN_MODE: '{RUN_MODE}'. Use: single | batch | evaluate_all")

  Model loaded : ./radiomic_output/brain_tumor_classifier.pkl
  Classes      : ['glioma', 'healthy', 'meningioma', 'pituitary']
  Feature cols : 53

  BATCH INFERENCE : GLIOMA
  Patients        : 335
  True label      : glioma
  Output          : glioma_inference_20260528_0539.csv



glioma:   0%|          | 1/335 [00:26<2:28:38, 26.70s/it]

  [  1/335] BraTS19_2013_10_1                        GLIOMA       [high]  ✓


glioma:   1%|          | 2/335 [00:39<1:44:17, 18.79s/it]

  [  2/335] BraTS19_2013_11_1                        GLIOMA       [high]  ✓


glioma:   1%|          | 2/335 [00:40<1:51:57, 20.17s/it]


KeyboardInterrupt: 